<p class="lead">Este notebook es la edición en R de la tercera unidad del curso. Las diapositivas de clase son un mapa; este texto es el territorio. La Unidad 2 introdujo el aprendizaje a partir de datos y un primer clasificador, la regresión logística, que traza una frontera recta a través del espacio de características. Esta unidad presenta otras tres familias de clasificadores que abordan el mismo problema desde tres direcciones distintas: los métodos <em>probabilísticos</em>, que modelan cómo se generan los datos de cada clase y luego invierten el modelo con el teorema de Bayes (Bayes ingenuo, redes bayesianas); los métodos <em>geométricos</em>, que buscan directamente la frontera con el mayor margen de seguridad (máquinas de vectores de soporte); y los métodos <em>basados en memoria</em>, que almacenan la muestra de entrenamiento y clasifican por analogía con los casos almacenados más cercanos (k vecinos más cercanos). Cada método se deriva primero, luego se implementa a mano en unas pocas líneas de R, después se compara con la implementación de un paquete y, por último, se pone en práctica con los datos de cáncer de mama que reaparecen a lo largo de todo el curso. La Sección 8 compara las tres familias de forma honesta, con la metodología del Suplemento S1.</p>

<div class="box key"><h4>Objetivos de aprendizaje</h4>
<ul class="objectives">
<li>Enunciar el teorema de Bayes, aplicarlo a mano a una prueba diagnóstica y explicar por qué un resultado positivo de una prueba exacta para una enfermedad poco frecuente suele ser un falso positivo.</li>
<li>Definir la probabilidad a posteriori, el clasificador de Bayes y el error de Bayes, y mostrar cómo los costos desiguales de clasificación errónea desplazan el umbral de decisión.</li>
<li>Derivar el clasificador de Bayes ingenuo gaussiano a partir del supuesto de independencia condicional, implementarlo en R base y explicar por qué clasifica bien incluso cuando el supuesto es falso.</li>
<li>Leer una red bayesiana como una factorización de una distribución conjunta, realizar inferencia por enumeración en una pequeña red médica y explicar cómo la evidencia y la d-separación modifican las probabilidades a posteriori.</li>
<li>Describir la regla de los k vecinos más cercanos, sus métricas de distancia, su dependencia del escalado de las características y de <em>k</em>, su costo computacional y la maldición de la dimensionalidad.</li>
<li>Explicar el principio de margen máximo, el papel de los vectores de soporte, del parámetro <em>C</em> y del kernel, y la relación entre la pérdida bisagra (hinge loss) y la pérdida logística de la Unidad 2.</li>
<li>Comparar las tres familias sobre los mismos datos con validación cruzada estratificada y elegir entre ellas según sus supuestos, costos e interpretabilidad.</li>
</ul></div>

<div class="box bridge"><h4>Cómo leer este notebook</h4>
<p>Cada bloque de código (chunk) es breve y está comentado. Ejecútalo, lee su salida y luego lee el párrafo que interpreta esa salida. Los ejercicios (cajas verdes) te piden modificar el bloque de código que está sobre ellos y volver a ejecutarlo; son la forma más rápida de aprender. Las cajas azules contienen ideas clave, las naranjas errores frecuentes y las moradas conectan esta unidad con otras unidades o con el curso de bioinformática (BGM324). El notebook usa solo R base junto con <code>mlbench</code>, <code>e1071</code>, <code>class</code>, <code>MASS</code> y <code>pROC</code>. En un entorno de ejecución de R recién iniciado en Google Colab, el estudiante ejecuta una sola vez <code>install.packages(c("e1071","randomForest","rpart.plot","neuralnet","cluster","mlbench","pROC"))</code>; no se descarga ningún dato. Cuatro algoritmos se recorren a mano con ejemplos del tamaño de una hoja de papel (Secciones 2.2, 4.2–4.3, 5.2 y 6.1) antes de que el código los confirme. La metodología de evaluación —por qué se reservan casos, qué es la validación cruzada, qué mide el AUC— se da por conocida del Suplemento S1 y no se vuelve a derivar aquí.</p></div>

<div class="box warn"><h4>En qué se diferencia la edición en R del curso en Python</h4><p>El curso en Python usa <code>load_breast_cancer</code> de scikit-learn: 569 aspirados y 30 características nucleares continuas. La versión espejo en R usa los datos Wisconsin incluidos en <code>mlbench</code>,<sup class="c"><a href="#r4">[4]</a>,<a href="#r40">[40]</a></sup> que corresponden al conjunto Wisconsin <em>original</em>: 683 casos completos descritos por nueve características puntuadas como enteros de 1 a 10. Los métodos, el razonamiento y las conclusiones son idénticos; solo cambian el número y el tipo de características, y donde el texto de Python dice "30 características" esta edición dice "nueve". Las demostraciones de regresión usan <code>BostonHousing</code> (una variable objetivo continua) en lugar de los datos de diabetes de scikit-learn, y las demostraciones con dígitos usan pequeñas matrices de imágenes <em>simuladas</em>, porque R no tiene un conjunto de dígitos incorporado. Cada sustitución se señala donde aparece.</p></div>



In [ ]:
# Bibliotecas usadas en todo el notebook. Las semillas se fijan para que cada número citado en el texto sea reproducible.
suppressPackageStartupMessages({
  library(mlbench); library(e1071); library(class); library(MASS); library(pROC)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#7f7f7f"  # benigno, maligno, acento, neutro

# Pequeños generadores que reproducen las demostraciones make_blobs / make_moons / make_circles de scikit-learn.
make_blobs <- function(n, centers, sd = 0.6) {
  k <- nrow(centers); per <- n %/% k
  X <- do.call(rbind, lapply(seq_len(k), function(i) MASS::mvrnorm(per, centers[i, ], diag(sd^2, 2))))
  list(X = X, y = factor(rep(c("neg", "pos"), length.out = k)[rep(seq_len(k), each = per)]))
}
make_moons <- function(n = 300, noise = 0.25) {
  n1 <- n %/% 2; n2 <- n - n1; t1 <- seq(0, pi, length.out = n1); t2 <- seq(0, pi, length.out = n2)
  X <- rbind(cbind(cos(t1), sin(t1)), cbind(1 - cos(t2), 0.5 - sin(t2))) + matrix(rnorm(2 * n, 0, noise), ncol = 2)
  list(X = X, y = factor(rep(c("A", "B"), c(n1, n2))))
}
make_circles <- function(n = 300, noise = 0.1, factor = 0.4) {
  n1 <- n %/% 2; n2 <- n - n1; to <- seq(0, 2 * pi, length.out = n1); ti <- seq(0, 2 * pi, length.out = n2)
  X <- rbind(cbind(cos(to), sin(to)), cbind(factor * cos(ti), factor * sin(ti))) + matrix(rnorm(2 * n, 0, noise), ncol = 2)
  list(X = X, y = factor(rep(c("outer", "inner"), c(n1, n2))))
}

# Sombrea las regiones de decisión de cualquier clasificador de 2 características, dada una función que etiqueta una grilla como 0/1 (1 = positivo).
shade_regions <- function(predfun, xlim, ylim, cn = c("x", "y"), res = 160, main = "", xlab = cn[1], ylab = cn[2]) {
  gx <- seq(xlim[1], xlim[2], length.out = res); gy <- seq(ylim[1], ylim[2], length.out = res)
  G <- setNames(expand.grid(gx, gy), cn)
  Z <- matrix(as.integer(predfun(G)), nrow = res)
  image(gx, gy, Z, col = c("#dfe7f1", "#fbe3d3"), main = main, xlab = xlab, ylab = ylab)
}
cat("R", R.version.string, "| e1071", as.character(packageVersion("e1071")),
    "| class", as.character(packageVersion("class")), "| mlbench", as.character(packageVersion("mlbench")), "\n")



La salida registra las versiones de R<sup class="c"><a href="#r36">[36]</a></sup> y de los paquetes con que se construyeron estos apuntes, como exige el hábito de reproducibilidad del Suplemento S0.

## 1. Tres maneras de trazar una frontera

La Unidad 1 definió un clasificador como una regla que divide el espacio de características en regiones y asigna una clase a cada una, y la Unidad 2 construyó la primera regla de ese tipo, la regresión logística, cuyas regiones están separadas por una línea recta. Hay muchas maneras de llegar a una frontera, y los métodos de esta unidad representan tres de ellas.

1. **Los métodos probabilísticos (generativos)** se preguntan cómo se distribuyen los datos de cada clase —"¿cómo son las características de los tumores malignos, y las de los benignos?"—, estiman esas distribuciones a partir de la muestra de entrenamiento y luego usan el teorema de Bayes para convertirlas en la probabilidad de cada clase dadas las características. La frontera es el conjunto de puntos donde dos clases son igualmente probables. Bayes ingenuo (Sección 4) y las redes bayesianas (Sección 5) pertenecen a este grupo.
2. **Los métodos geométricos (discriminativos)** no modelan las clases en absoluto. Buscan directamente la frontera que separa los casos de entrenamiento y, entre todas las fronteras que lo logran, eligen la que deja el corredor vacío más ancho —el <em>margen</em>— entre las clases. La máquina de vectores de soporte (Sección 7) es el ejemplo canónico. La regresión logística también es discriminativa, pero elige su recta con un criterio distinto.
3. **Los métodos basados en memoria (basados en instancias)** no construyen ningún modelo ni ninguna frontera explícita. Almacenan la muestra de entrenamiento y, cuando llega un caso nuevo, buscan los casos almacenados más parecidos y copian su etiqueta. La regla de los k vecinos más cercanos (Sección 6) es el arquetipo; su frontera existe solo de forma implícita, como los lugares donde cambia la mayoría entre los vecinos.

La distinción importa porque las tres familias fallan de maneras distintas. Un modelo generativo es tan bueno como sus supuestos sobre las distribuciones de las clases; un modelo basado en el margen es tan bueno como su elección de la forma de la frontera; un modelo basado en memoria es tan bueno como su noción de similitud y la densidad de casos almacenados. El mismo problema de dos características se resolverá con las tres para que se vean las diferencias. Los datos son, como en todas las unidades, la muestra de cáncer de mama de Wisconsin.<sup class="c"><a href="#r4">[4]</a></sup> Para las imágenes bidimensionales conservamos las dos características usadas a mano en la Unidad 1, <em>clump thickness</em> (grosor del grumo) y <em>cell size</em> (tamaño celular); el problema completo de nueve características vuelve en la Sección 8. Una introducción a todos estos métodos en R, al nivel de este curso, es el libro de texto de James y colaboradores.<sup class="c"><a href="#r39">[39]</a></sup>



In [ ]:
# El ejemplo recurrente: dos características de los datos de cáncer de mama, divididos una vez en 70% entrenamiento / 30% prueba.
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                  # elimina la columna Id
bc <- bc[complete.cases(bc), ]                            # elimina los 16 registros incompletos
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))   # factor ordinal -> numérico 1..10
bc$Class <- factor(bc$Class, levels = c("benign", "malignant"))
feat2 <- c("Cl.thickness", "Cell.size")
n <- nrow(bc); set.seed(0); idx <- sample(n, round(0.7 * n))
tr <- bc[idx, ]; te <- bc[-idx, ]
cat(sprintf("training cases: %d (benign %d, malignant %d) | test cases: %d\n",
            nrow(tr), sum(tr$Class == "benign"), sum(tr$Class == "malignant"), nrow(te)))
pcol <- function(d) ifelse(d$Class == "malignant", ORANGE, BLUE)
plot(jitter(tr[[feat2[1]]]), jitter(tr[[feat2[2]]]), col = pcol(tr), pch = 19, cex = 0.7,
     xlab = "clump thickness (1-10)", ylab = "cell size (1-10)",
     main = "The two-feature problem that all three families will solve")
legend("topleft", c("malignant", "benign"), col = c(ORANGE, BLUE), pch = 19, bty = "n")



<p class="caption"><b>Figura 1.</b> Los 478 casos de entrenamiento en el plano del grosor del grumo y el tamaño celular (con una pequeña perturbación aleatoria añadida —jitter—, porque ambas características son puntuaciones enteras de 1 a 10). Las clases son separables en la mayor parte —los tumores benignos se agrupan en puntuaciones bajas—, pero se superponen en una franja de valores intermedios; ninguna frontera, de ninguna forma, clasificará correctamente todos los casos, y la pregunta para cada método es dónde, y con qué forma, trazar la línea a través de la superposición.</p>

El bloque de código siguiente muestra la respuesta que da cada una de las tres familias, con la configuración por defecto, antes de haber explicado ninguna. La figura es un adelanto al que conviene volver: al final de la unidad, cada rasgo de los tres paneles —la frontera curva de Bayes ingenuo, la franja recta de la máquina de vectores de soporte lineal, las islas irregulares de los k vecinos más cercanos— tendrá una explicación.



In [ ]:
# Adelanto: el mismo problema resuelto por un clasificador probabilístico, uno geométrico y uno basado en memoria.
f2 <- as.formula(paste("Class ~", paste(feat2, collapse = " + ")))
nb_p  <- naiveBayes(f2, data = tr)
svm_p <- svm(f2, data = tr, kernel = "linear", cost = 1)
ctr <- colMeans(tr[feat2]); csd <- apply(tr[feat2], 2, sd)               # escalado para k-NN
sc <- function(df) scale(df[feat2], center = ctr, scale = csd)
xlim <- c(0.5, 10.5); ylim <- c(0.5, 10.5)
preds <- list(
  "Probabilistic: Gaussian naive Bayes" = function(G) predict(nb_p, G) == "malignant",
  "Geometric: linear SVM"               = function(G) predict(svm_p, G) == "malignant",
  "Memory-based: 5 nearest neighbours"  = function(G) knn(sc(tr), scale(G, center = ctr, scale = csd), tr$Class, k = 5) == "malignant")
accs <- c(mean(predict(nb_p, te) == te$Class), mean(predict(svm_p, te) == te$Class),
          mean(knn(sc(tr), sc(te), tr$Class, k = 5) == te$Class))
par(mfrow = c(1, 3), mar = c(4, 4, 3, 1))
for (i in seq_along(preds)) {
  shade_regions(preds[[i]], xlim, ylim, cn = feat2, res = 140,
                main = sprintf("%s\ntest accuracy %.3f", names(preds)[i], accs[i]),
                xlab = "clump thickness", ylab = if (i == 1) "cell size" else "")
  points(jitter(te[[feat2[1]]]), jitter(te[[feat2[2]]]), col = pcol(te), pch = 19, cex = 0.6)
}



<p class="caption"><b>Figura 2.</b> Regiones de decisión de las tres familias sobre los 205 casos de prueba, cada una ajustada con la configuración por defecto a los 478 casos de entrenamiento. Las exactitudes de los títulos están todas cerca de 0.93 —el problema es lo bastante fácil como para que cualquier regla sensata funcione bien—, pero las fronteras difieren en su forma: Bayes ingenuo traza una curva suave (una sección cónica, Sección 4.4), la máquina de vectores de soporte lineal una recta ubicada para maximizar la separación entre las clases (Sección 7.1), y k vecinos más cercanos una frontera irregular que sigue a la mayoría local (Sección 6.4). Las diferencias son invisibles en la exactitud y decisivas en cómo se comporta cada método lejos de los datos.</p>

## 2. La probabilidad necesaria para esta unidad

Dos de las tres familias se construyen sobre la probabilidad, y la tercera se entiende mejor a través de ella (Sección 6.2). Esta sección repasa solo los tres hechos necesarios e introduce el ejemplo diagnóstico recurrente que se resolverá a mano.

### 2.1 Probabilidad conjunta, marginal y condicional

Para dos eventos \(A\) y \(B\) —digamos \(D\), "el paciente tiene la enfermedad", y \(T\), "la prueba es positiva"—, la probabilidad **conjunta** \(P(D, T)\) es la probabilidad de que ocurran ambos; la probabilidad **marginal** \(P(T)\) es la probabilidad de \(T\) sin importar \(D\), que se obtiene sumando la conjunta sobre las posibilidades de \(D\) (la <em>ley de probabilidad total</em>, \(P(T) = P(T, D) + P(T, \neg D)\)); y la probabilidad **condicional** \(P(T \mid D)\) es la probabilidad de \(T\) entre los casos en que se cumple \(D\), definida por

\[ P(T \mid D) = \frac{P(T, D)}{P(D)} \qquad\text{equivalently}\qquad P(T, D) = P(T \mid D)\, P(D). \]

La segunda forma es la <em>regla del producto</em>, y es simétrica: también \(P(T, D) = P(D \mid T)\, P(T)\). Dos eventos son **independientes** cuando \(P(T \mid D) = P(T)\), es decir, cuando conocer uno no dice nada sobre el otro, en cuyo caso la conjunta se factoriza como \(P(T, D) = P(T)\,P(D)\). Dos eventos son **condicionalmente independientes dado** un tercero, \(C\), cuando \(P(A, B \mid C) = P(A \mid C)\, P(B \mid C)\): dentro de cada valor de \(C\) no aportan información uno sobre el otro, aunque puedan estar fuertemente asociados en conjunto. Esta última noción es la base tanto de Bayes ingenuo como de las redes bayesianas.

### 2.2 El teorema de Bayes, a mano

Igualar las dos formas de la regla del producto y dividir por \(P(T)\) da el resultado que Thomas Bayes dejó en un manuscrito publicado póstumamente en 1763:<sup class="c"><a href="#r1">[1]</a></sup>

<div class="eq">\[ P(D \mid T) = \frac{P(T \mid D)\, P(D)}{P(T)} = \frac{P(T \mid D)\, P(D)}{P(T \mid D)\, P(D) + P(T \mid \neg D)\, P(\neg D)} \]
<span class="where">donde \(P(D)\) es la probabilidad <strong>a priori</strong> de la enfermedad antes de la prueba (en un contexto de tamizaje, la prevalencia), \(P(T \mid D)\) es la <strong>verosimilitud</strong> del resultado observado bajo la enfermedad (para una prueba positiva, la sensibilidad), \(P(T)\) es la <strong>evidencia</strong> o probabilidad marginal del resultado observado, y \(P(D \mid T)\) es la probabilidad <strong>a posteriori</strong> de la enfermedad después de la prueba; en lenguaje clínico, el valor predictivo positivo (VPP, en inglés PPV).</span></div>

El teorema es una identidad; su contenido está en la dirección de la inferencia. Las características de una prueba se miden en la dirección directa (¿cómo se comporta la prueba en personas que se sabe que están enfermas?), mientras que el clínico necesita la dirección inversa (¿qué tan probable es la enfermedad en una persona cuya prueba es positiva?). Las dos direcciones se relacionan solo a través de la probabilidad a priori, y el descuido de la probabilidad a priori —la <em>falacia de la tasa base</em> (base-rate neglect)— es un error bien documentado: cuando Casscells y colaboradores plantearon a personal y estudiantes de la Facultad de Medicina de Harvard una versión del problema siguiente, menos de uno de cada cinco dio la respuesta correcta, y la respuesta más frecuente estaba errada por un factor de cinco.<sup class="c"><a href="#r2">[2]</a></sup>

**Ejemplo resuelto.** Una prueba de tamizaje para una enfermedad con prevalencia del 1 % tiene sensibilidad del 95 % (\(P(T \mid D) = 0.95\)) y especificidad del 95 % (\(P(\neg T \mid \neg D) = 0.95\), por lo que la tasa de falsos positivos es \(P(T \mid \neg D) = 0.05\)). Una persona obtiene un resultado positivo. ¿Cuál es la probabilidad de que tenga la enfermedad?

\[ P(D \mid T) = \frac{0.95 \times 0.01}{0.95 \times 0.01 + 0.05 \times 0.99} = \frac{0.0095}{0.0095 + 0.0495} = \frac{0.0095}{0.0590} \approx 0.161. \]

Un resultado positivo de una prueba que acierta el 95 % de las veces en ambas direcciones deja al paciente con solo un 16 % de probabilidad de enfermedad, porque el 1 % de casos verdaderos que dan positivo (0.95 × 1 % ≈ 0.95 % del total) es superado cinco a uno por el 99 % de personas sanas que dan falsamente positivo (0.05 × 99 % ≈ 4.95 % del total). La misma aritmética es mucho más fácil de seguir en <em>frecuencias naturales</em> —conteos en una población hipotética de 10,000 personas— que en probabilidades, como demostraron experimentalmente Gigerenzer y Hoffrage,<sup class="c"><a href="#r3">[3]</a></sup> y el bloque de código siguiente la presenta de ambas formas.



In [ ]:
# Teorema de Bayes en el ejemplo de tamizaje, como probabilidades y como frecuencias naturales en 10,000 personas.
prevalence <- 0.01; sensitivity <- 0.95; specificity <- 0.95
p_pos_D <- sensitivity; p_pos_notD <- 1 - specificity
evidence <- p_pos_D * prevalence + p_pos_notD * (1 - prevalence)        # P(T) por probabilidad total
ppv <- p_pos_D * prevalence / evidence
cat(sprintf("P(T) = %.4f   P(D | T) = PPV = %.4f\n", evidence, ppv))
N <- 10000; ill <- N * prevalence; healthy <- N * (1 - prevalence)
freq <- rbind("ill (100)"      = c(`test +` = ill * sensitivity,      `test -` = ill * (1 - sensitivity)),
              "healthy (9,900)" = c(`test +` = healthy * (1 - specificity), `test -` = healthy * specificity))
freq <- round(freq); print(freq)
cat(sprintf("Of the %d positives, %d are ill: %.3f\n", sum(freq[, 1]), freq[1, 1], freq[1, 1] / sum(freq[, 1])))
prev <- seq(0.001, 0.6, length.out = 400)
plot(prev * 100, 0.95 * prev / (0.95 * prev + 0.05 * (1 - prev)), type = "l", col = BLUE, lwd = 2,
     ylim = c(0, 1), xlab = "prevalence (%)", ylab = "P(disease | positive test)",
     main = "The posterior depends on the prior: PPV vs prevalence")
lines(prev * 100, 0.95 * prev / (0.95 * prev + 0.01 * (1 - prev)), col = GREEN, lwd = 2)
lines(prev * 100, 0.95 * prev / (0.95 * prev + 0.001 * (1 - prev)), col = ORANGE, lwd = 2)
abline(v = 1, col = GREY, lty = 3); points(1, ppv, pch = 19, col = BLUE)
legend("bottomright", c("specificity 0.95", "specificity 0.99", "specificity 0.999"),
       col = c(BLUE, GREEN, ORANGE), lwd = 2, bty = "n")
for (p in c(0.01, 0.10, 0.30))
  cat(sprintf("prevalence %2.0f%%: PPV = %.3f (specificity 0.95)\n", p * 100, 0.95 * p / (0.95 * p + 0.05 * (1 - p))))



El código reproduce el cálculo a mano: el VPP es 0.161. En la tabla de frecuencias, 95 de las 100 personas enfermas y 495 de las 9,900 personas sanas dan positivo, de modo que 95 de los 590 positivos están enfermos. La misma prueba aplicada en una sala hospitalaria donde la prevalencia es del 30 % en lugar del 1 % se comportaría de manera muy distinta; la figura recorre el VPP en todo el rango de prevalencias y muestra además lo que se gana con una prueba más específica.

<p class="caption"><b>Figura 3.</b> Valor predictivo positivo frente a prevalencia para una prueba con sensibilidad 0.95 y tres especificidades. Con 1 % de prevalencia, la prueba con 95 % de especificidad da un VPP de 0.16; con 10 % da alrededor de 0.68 y con 30 % alrededor de 0.89 (valores impresos). Subir la especificidad a 0.999 hace que la prueba sea útil incluso en tamizaje. La lección se traslada directamente a los clasificadores: la sensibilidad y la especificidad de un modelo son propiedades del modelo, pero el significado de su salida positiva depende de la población a la que se aplica, lo cual es una de las razones por las que los fracasos de validación externa de la Unidad 1 ocurrieron en hospitales con una composición de casos distinta.</p>

<div class="box warn"><h4>Error frecuente</h4><p>"La prueba tiene un 95 % de exactitud, así que un resultado positivo significa un 95 % de probabilidad de enfermedad". El 95 % es \(P(T \mid D)\); el clínico necesita \(P(D \mid T)\). Ambos son iguales solo cuando la probabilidad a priori es del 50 %. Todo clasificador probabilístico de esta unidad entrega una probabilidad a posteriori, y toda probabilidad a posteriori hereda la probabilidad a priori que se incorporó en ella, a través de las proporciones de clase de la muestra de entrenamiento. Un modelo de Bayes ingenuo entrenado con una muestra que es mitad maligna sobrestimará la probabilidad de malignidad en una población de tamizaje; la Sección 3.1 muestra dónde entra la probabilidad a priori.</p></div>

### 2.3 El teorema de Bayes para clasificación

Reemplazar el evento binario \(D\) por una etiqueta de clase \(y \in \{1, \dots, K\}\) y el resultado de la prueba por un vector de características \(\mathbf{x}\) da la forma que se usa en toda esta unidad:

\[ P(y = c \mid \mathbf{x}) = \frac{p(\mathbf{x} \mid y = c)\, P(y = c)}{p(\mathbf{x})} = \frac{p(\mathbf{x} \mid y = c)\, P(y = c)}{\sum_{k=1}^{K} p(\mathbf{x} \mid y = k)\, P(y = k)}. \]

La \(p\) minúscula indica una <em>densidad</em> de probabilidad, porque las características suelen ser continuas: \(p(\mathbf{x} \mid y = c)\) es la <strong>densidad condicional de clase</strong> —la distribución de las características entre los casos de la clase \(c\)— y \(P(y = c)\) es la <strong>probabilidad a priori de la clase</strong>. Un clasificador generativo es una receta para estimar estos dos objetos a partir de la muestra de entrenamiento; una vez que se tienen, la probabilidad a posteriori se obtiene de la fórmula, y el denominador puede ignorarse al comparar clases porque es el mismo para todas ellas.

## 3. Teoría bayesiana de la decisión

### 3.1 El clasificador de Bayes y el error de Bayes

Supongamos por un momento que las densidades condicionales de clase y las probabilidades a priori se conocen exactamente. Entonces la probabilidad a posteriori \(P(y = c \mid \mathbf{x})\) se conoce para todo \(\mathbf{x}\), y la regla que asigna cada \(\mathbf{x}\) a la clase de mayor probabilidad a posteriori,

\[ \hat{y}(\mathbf{x}) = \arg\max_c \, P(y = c \mid \mathbf{x}) = \arg\max_c \, p(\mathbf{x} \mid y = c)\, P(y = c), \]

es el **clasificador de Bayes**. Es óptimo en un sentido preciso: ninguna otra regla tiene menor probabilidad de error. Su error, el **error de Bayes**, es el error irreducible del problema: la parte causada por la superposición genuina de las clases en el espacio de características, que ninguna cantidad de datos ni de modelado puede eliminar. El lugar donde las dos clases tienen igual probabilidad a posteriori, \(P(y = c \mid \mathbf{x}) = 1/2\) en el caso binario, es la **frontera de decisión de Bayes**; todo clasificador de este curso puede leerse como un intento de aproximarla a partir de una muestra finita. La probabilidad a priori entra como un peso multiplicativo sobre cada clase: duplicar la probabilidad a priori de malignidad duplica sus odds a posteriori en todas partes y desplaza la frontera hacia el lado benigno.

### 3.2 Decisiones de mínimo error y de mínimo riesgo

Elegir la clase más probable minimiza el <em>número</em> de errores. En medicina, los dos tipos de error rara vez cuestan lo mismo: un tumor maligno clasificado como benigno es un cáncer no detectado; uno benigno clasificado como maligno es una biopsia innecesaria. Sea \(\lambda_{\mathrm{FN}}\) el costo de un falso negativo (decidir benigno cuando la verdad es maligno) y \(\lambda_{\mathrm{FP}}\) el costo de un falso positivo. El **riesgo esperado** de cada decisión en un punto \(\mathbf{x}\) es

\[ R(\text{decide benign} \mid \mathbf{x}) = \lambda_{\mathrm{FN}}\, P(M \mid \mathbf{x}), \qquad R(\text{decide malignant} \mid \mathbf{x}) = \lambda_{\mathrm{FP}}\, P(B \mid \mathbf{x}), \]

y la regla de **mínimo riesgo** decide maligno cuando el primero supera al segundo, es decir, cuando

\[ P(M \mid \mathbf{x}) > \frac{\lambda_{\mathrm{FP}}}{\lambda_{\mathrm{FP}} + \lambda_{\mathrm{FN}}}. \]

Con costos iguales el umbral es \(1/2\) y la regla se reduce a la de mínimo error. Si se juzga que un cáncer no detectado es cinco veces más costoso que una biopsia innecesaria, el umbral baja a \(1/6 \approx 0.167\): el clasificador declara maligno un caso en cuanto la probabilidad a posteriori supera un sexto. El bloque de código siguiente construye toda la maquinaria sobre una sola característica, el grosor del grumo (clump thickness), ajustando una gaussiana por clase y usando como probabilidades a priori las proporciones de clase de la muestra de entrenamiento.



In [ ]:
# Teoría bayesiana de la decisión en una dimensión: gaussianas condicionales de clase del grosor del grumo, probabilidades a priori, a posteriori, umbrales.
r_tr <- tr$Cl.thickness
prior <- c(malignant = mean(tr$Class == "malignant"), benign = mean(tr$Class == "benign"))
mu <- c(malignant = mean(r_tr[tr$Class == "malignant"]), benign = mean(r_tr[tr$Class == "benign"]))
sdv <- c(malignant = sd(r_tr[tr$Class == "malignant"]),  benign = sd(r_tr[tr$Class == "benign"]))
grid <- seq(0, 12, length.out = 1000)
jM <- dnorm(grid, mu["malignant"], sdv["malignant"]) * prior["malignant"]   # p(x|M) P(M)
jB <- dnorm(grid, mu["benign"],    sdv["benign"])    * prior["benign"]
postM <- jM / (jM + jB)
t_err  <- grid[which.min(abs(postM - 0.5))]
t_risk <- grid[which.min(abs(postM - 1/6))]
cat(sprintf("priors: malignant %.3f, benign %.3f | means: %.2f vs %.2f, sd: %.2f vs %.2f\n",
            prior["malignant"], prior["benign"], mu["malignant"], mu["benign"], sdv["malignant"], sdv["benign"]))
cat(sprintf("clump-thickness threshold: minimum-error %.2f | minimum-risk (missed cancer 5x costlier) %.2f\n", t_err, t_risk))
par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
plot(grid, jM, type = "l", col = ORANGE, lwd = 2, xlab = "clump thickness", ylab = "density x prior",
     main = "Class-conditional densities x priors", ylim = c(0, max(jM, jB)))
lines(grid, jB, col = BLUE, lwd = 2)
polygon(c(grid, rev(grid)), c(pmin(jM, jB), rep(0, length(grid))), col = adjustcolor(GREY, 0.35), border = NA)
legend("topright", c("p(x|M) P(M)", "p(x|B) P(B)", "Bayes error (overlap)"),
       col = c(ORANGE, BLUE, GREY), lwd = c(2, 2, 6), bty = "n", cex = 0.8)
plot(grid, postM, type = "l", col = ORANGE, lwd = 2, xlab = "clump thickness", ylab = "P(malignant | x)",
     main = "Posterior and the two decision thresholds")
abline(h = 0.5, lty = 3); abline(v = t_err, lty = 2)
abline(h = 1/6, col = GREEN, lty = 3); abline(v = t_risk, col = GREEN, lty = 2)
legend("bottomright", c(sprintf("equal costs: %.1f", t_err), sprintf("FN costs 5x FP: %.1f", t_risk)),
       col = c("black", GREEN), lty = 2, bty = "n", cex = 0.8)
# Consecuencias en los 205 casos de prueba: errores de cada tipo y costo total (FN = 5, FP = 1).
rows <- lapply(list(c("minimum error (threshold 1/2)", t_err), c("minimum risk (threshold 1/6)", t_risk)),
  function(z) {
    t <- as.numeric(z[2]); callM <- te$Cl.thickness >= t
    FN <- sum(!callM & te$Class == "malignant"); FP <- sum(callM & te$Class == "benign")
    data.frame(rule = z[1], FN = FN, FP = FP, errors = FN + FP,
               sens = round(mean(callM[te$Class == "malignant"]), 3),
               spec = round(mean(!callM[te$Class == "benign"]), 3), cost_5FN_FP = 5 * FN + FP)
  })
print(do.call(rbind, rows), row.names = FALSE)



<p class="caption"><b>Figura 4.</b> Izquierda: las dos gaussianas condicionales de clase del grosor del grumo, cada una multiplicada por su probabilidad a priori, de modo que las curvas son proporcionales a la probabilidad a posteriori; la superposición sombreada es la región en la que el clasificador de Bayes necesariamente se equivoca. Derecha: la probabilidad a posteriori de malignidad. El umbral de costos iguales está donde la probabilidad a posteriori cruza 0.5; cuando un cáncer no detectado es cinco veces más costoso que una biopsia innecesaria, el umbral se desplaza a donde la probabilidad a posteriori cruza 1/6 (valores impresos), y más casos se declaran malignos. La gaussiana maligna es más ancha que la benigna (desviaciones estándar impresas), de modo que lejos de ambas medias sus colas superan a la densidad benigna —el mismo efecto de la varianza que decide el ejemplo a mano de la Sección 4.3—; en la práctica nadie declararía maligno un núcleo muy delgado y muy regular, lo que recuerda que un modelo solo es confiable donde hay datos.</p>

Los dos umbrales intercambian un tipo de error por el otro. La tabla impresa sobre la figura aplica ambos a los 205 casos de prueba y cuenta las consecuencias, incluido el costo total bajo el supuesto 5 : 1. La regla de mínimo error comete menos errores en total; la regla de mínimo riesgo comete más errores, pero menos del tipo costoso, y su costo total bajo los costos establecidos es menor. Ninguna de las dos reglas es "mejor" en abstracto: la elección de los costos es una decisión clínica y ética que el ingeniero debe recabar y hacer explícita, no un detalle de modelado. El Suplemento S1 generaliza este compromiso en la curva ROC, que muestra todos los umbrales a la vez.

<div class="box key"><h4>Idea clave</h4><p>La teoría bayesiana de la decisión separa dos cosas que a menudo se confunden: <em>estimar</em> la probabilidad a posteriori de cada clase, que es un problema estadístico, y <em>decidir</em> qué hacer con ella, lo cual depende de los costos de los posibles errores. Un clasificador que entrega probabilidades calibradas puede servir a cualquier estructura de costos moviendo el umbral; uno que entrega solo etiquetas lleva incorporado el supuesto de costos iguales.</p></div>

## 4. Bayes ingenuo

### 4.1 El supuesto de independencia condicional

Para usar el teorema de Bayes en clasificación hay que estimar la densidad condicional de clase \(p(\mathbf{x} \mid y = c)\). Para una sola característica esto es fácil: basta un histograma o una gaussiana ajustada. Para muchas características es, en general, imposible, porque una densidad conjunta en nueve (o treinta) dimensiones tiene demasiados grados de libertad para estimarse a partir de unos pocos cientos de casos: es la maldición de la dimensionalidad que la Sección 6.5 cuantificará. El clasificador de **Bayes ingenuo** (naive Bayes) escapa suponiendo que, <em>dentro de cada clase</em>, las características son independientes entre sí:

<div class="eq">\[ p(\mathbf{x} \mid y = c) = \prod_{j=1}^{p} p(x_j \mid y = c), \qquad\text{so that}\qquad P(y = c \mid \mathbf{x}) \propto P(y = c) \prod_{j=1}^{p} p(x_j \mid y = c). \]
<span class="where">La densidad de p dimensiones se reemplaza por p densidades unidimensionales por clase, cada una de las cuales puede estimarse solo con los datos de esa clase. El clasificador necesita entonces únicamente la probabilidad a priori de cada clase y, para cada característica, un modelo unidimensional de su distribución en cada clase.</span></div>

El supuesto es casi siempre falso —en los datos de cáncer de mama, el tamaño celular y la forma celular de un mismo núcleo están fuertemente correlacionados dentro de cada clase—, y por eso el método se llama <em>ingenuo</em> o, en palabras de Hand y Yu, "Bayes del idiota" (idiot's Bayes).<sup class="c"><a href="#r6">[6]</a></sup> La Sección 4.6 explica por qué funciona de todos modos. En la práctica el producto se calcula como una suma de logaritmos, porque el producto de muchas densidades produce un desbordamiento por defecto (underflow) del rango de punto flotante:

\[ \log P(y = c \mid \mathbf{x}) = \log P(y = c) + \sum_{j=1}^{p} \log p(x_j \mid y = c) + \text{const}. \]

La elección del modelo unidimensional \(p(x_j \mid y = c)\) da lugar a las variantes del método. **Bayes ingenuo gaussiano**, para características continuas, ajusta una distribución normal \(\mathcal{N}(\mu_{cj}, \sigma^2_{cj})\) a cada característica en cada clase, de modo que el entrenamiento consiste en calcular una media y una varianza por característica y clase, más las proporciones de clase. **Bayes ingenuo multinomial**, para características de conteo como las frecuencias de palabras en un documento, modela cada clase mediante un vector de probabilidades sobre el vocabulario; **Bayes ingenuo de Bernoulli**, para características binarias (palabra presente o ausente), modela cada característica mediante una probabilidad por clase de valer 1. Los dos últimos son el filtro de spam clásico y el modelo de referencia (baseline) de la clasificación de textos, incluida la clasificación de notas clínicas; la Sección 4.7 los muestra con datos de imágenes. En R, las variantes gaussiana y categórica las proporciona <code>e1071::naiveBayes</code>;<sup class="c"><a href="#r37">[37]</a></sup> las variantes de conteo se escriben desde cero más adelante.

Bayes ingenuo tiene tres virtudes prácticas: es <strong>fácil de construir</strong> (sin optimización iterativa, sin hiperparámetros aparte de una constante de suavizado), es <strong>eficiente</strong> (el entrenamiento es una sola pasada sobre los datos y la predicción es una suma de \(p\) términos) y funciona bien con <strong>datos muy grandes y de alta dimensión</strong>, incluidos datos ruidosos, porque el número de parámetros crece solo linealmente con el número de características.

### 4.2 Bayes ingenuo a mano: el ejemplo del clima

El ejemplo clásico a mano, usado en las diapositivas de clase y en la mayoría de los libros de texto, es el conjunto de datos de catorce días "play tennis" de Quinlan: cuatro características categóricas —Outlook (pronóstico: Sunny / Overcast / Rainy, es decir, soleado / nublado / lluvioso), Temperature (temperatura: Hot / Mild / Cool, es decir, calurosa / templada / fresca), Humidity (humedad: High / Normal, alta / normal) y Windy (ventoso: True / False)— y una variable objetivo binaria, si se jugó el partido (Yes en 9 días, No en 5).<sup class="c"><a href="#r12">[12]</a></sup> El cálculo tiene tres pasos.

**Paso 1 — tabla de frecuencias.** Cuenta, para cada valor de Outlook, cuántos días cayeron en cada clase:

| Outlook | No | Yes | total |
|---|---|---|---|
| Overcast | 0 | 4 | 4 |
| Rainy | 3 | 2 | 5 |
| Sunny | 2 | 3 | 5 |
| **total** | **5** | **9** | **14** |

**Paso 2 — tabla de verosimilitudes.** Convierte los conteos en probabilidades: la probabilidad a priori \(P(\text{Yes}) = 9/14 = 0.64\), la evidencia \(P(\text{Sunny}) = 5/14 = 0.36\) y la verosimilitud \(P(\text{Sunny} \mid \text{Yes}) = 3/9 = 0.33\).

**Paso 3 — probabilidad a posteriori.** El teorema de Bayes da \(P(\text{Yes} \mid \text{Sunny}) = \dfrac{P(\text{Sunny} \mid \text{Yes})\, P(\text{Yes})}{P(\text{Sunny})} = \dfrac{0.33 \times 0.64}{0.36} = 0.60\), de modo que en un día soleado la predicción es **jugar**. (Exactamente, \(\tfrac{3}{9} \cdot \tfrac{9}{14} \big/ \tfrac{5}{14} = 3/5\).)

Con las cuatro características entra el paso "ingenuo": la verosimilitud del día completo es el producto de las verosimilitudes de cada característica. **Consulta:** Outlook = Rainy, Temperature = Mild, Humidity = Normal, Windy = True. Las probabilidades a posteriori no normalizadas son

\( P(\text{Yes}) \prod_j P(x_j \mid \text{Yes}) = \tfrac{9}{14} \times \tfrac{2}{9} \times \tfrac{4}{9} \times \tfrac{6}{9} \times \tfrac{3}{9} \approx 0.01411 \)

\( P(\text{No}) \prod_j P(x_j \mid \text{No}) = \tfrac{5}{14} \times \tfrac{3}{5} \times \tfrac{2}{5} \times \tfrac{1}{5} \times \tfrac{3}{5} \approx 0.01029 \)

y la **normalización** divide cada una por su suma, \(0.01411 + 0.01029 = 0.02440\) (que es \(p(\mathbf{x})\), la evidencia): \(P(\text{Yes} \mid \mathbf{x}) \approx 0.578\) y \(P(\text{No} \mid \mathbf{x}) \approx 0.422\). La predicción es **jugar**, con una confianza moderada. Observa el cero en la fila Overcast de la columna No: cualquier día nublado recibiría \(P(\text{No} \mid \mathbf{x}) = 0\) sin importar las otras tres características, porque un solo cero anula el producto. Este es el <em>problema de la frecuencia cero</em>, y el remedio estándar, el **suavizado de Laplace (aditivo)**, suma un pseudoconteo \(\alpha\) a cada celda antes de dividir. El bloque de código construye los catorce días, repite el cálculo en R base y luego con <code>e1071::naiveBayes</code>, primero sin suavizado (reproduciendo el resultado a mano) y luego con \(\alpha = 1\).

<div class="box warn"><h4>Dónde difieren las diapositivas de los datos originales</h4><p>La tabla de frecuencias anterior es la de las diapositivas de clase y la de muchos tutoriales en línea. En la tabla original de Quinlan, los conteos de Sunny y Rainy están al revés (Sunny: 2 Yes, 3 No; Rainy: 3 Yes, 2 No),<sup class="c"><a href="#r12">[12]</a></sup> de modo que la respuesta original para un día soleado es \(P(\text{Yes} \mid \text{Sunny}) = 2/5 = 0.40\): no jugar. La aritmética del método es idéntica; solo se intercambian las etiquetas de dos valores de Outlook. El notebook sigue la tabla de las diapositivas para que los números coincidan con la clase; el conjunto de datos que se construye a continuación es el de Quinlan con esas dos etiquetas intercambiadas.</p></div>



In [ ]:
# El conjunto de datos del clima de catorce días (tal como se tabula en las diapositivas), Bayes ingenuo a mano y luego con e1071.
weather <- data.frame(
  Outlook     = c("Rainy","Rainy","Overcast","Sunny","Sunny","Sunny","Overcast","Rainy","Rainy","Sunny","Rainy","Overcast","Overcast","Sunny"),
  Temperature = c("Hot","Hot","Hot","Mild","Cool","Cool","Cool","Mild","Cool","Mild","Mild","Mild","Hot","Mild"),
  Humidity    = c("High","High","High","High","Normal","Normal","Normal","High","Normal","Normal","Normal","High","Normal","High"),
  Windy       = factor(c(FALSE,TRUE,FALSE,FALSE,FALSE,TRUE,TRUE,FALSE,FALSE,FALSE,TRUE,TRUE,FALSE,TRUE)),
  Play        = c("No","No","Yes","Yes","Yes","No","Yes","No","Yes","Yes","Yes","Yes","Yes","No"),
  stringsAsFactors = TRUE)
cat("Step 1 - frequency table for Outlook:\n"); print(addmargins(table(Outlook = weather$Outlook, Play = weather$Play)))
prior <- prop.table(table(weather$Play))                               # P(Yes), P(No)
lik <- function(feature, value, cls) mean(weather[[feature]][weather$Play == cls] == value)
cat(sprintf("\nStep 2 - P(Sunny) = %.2f, P(Yes) = %.2f, P(Sunny | Yes) = %.2f\n",
            mean(weather$Outlook == "Sunny"), prior["Yes"], lik("Outlook", "Sunny", "Yes")))
cat(sprintf("Step 3 - P(Yes | Sunny) = %.2f\n\n", lik("Outlook", "Sunny", "Yes") * prior["Yes"] / mean(weather$Outlook == "Sunny")))
q <- list(Outlook = "Rainy", Temperature = "Mild", Humidity = "Normal", Windy = "TRUE")
unnorm <- sapply(c("Yes", "No"), function(c) prior[[c]] * prod(mapply(function(f, v) lik(f, v, c), names(q), q)))
cat("by hand, query (Rainy, Mild, Normal, Windy):\n")
for (c in c("Yes", "No")) cat(sprintf("  P(%s) x likelihoods = %.5f  ->  normalised P(%s | x) = %.3f\n",
                                       c, unnorm[c], c, unnorm[c] / sum(unnorm)))
qdf <- data.frame(Outlook = factor("Rainy", levels(weather$Outlook)), Temperature = factor("Mild", levels(weather$Temperature)),
                  Humidity = factor("Normal", levels(weather$Humidity)), Windy = factor("TRUE", levels(weather$Windy)))
cat(sprintf("\ne1071 naiveBayes: alpha=0 (laplace=0) P(Yes|x) = %.3f  |  alpha=1 (laplace=1) P(Yes|x) = %.3f\n",
            predict(naiveBayes(Play ~ ., weather, laplace = 0), qdf, type = "raw", threshold = 0)[1, "Yes"],
            predict(naiveBayes(Play ~ ., weather, laplace = 1), qdf, type = "raw")[1, "Yes"]))



Con \(\alpha = 0\), <code>e1071::naiveBayes</code> devuelve el resultado a mano, 0.578. Con \(\alpha = 1\) la probabilidad a posteriori cambia levemente a 0.586, porque cada verosimilitud se acerca a la uniforme y, más importante aún, la imposible \(P(\text{Overcast} \mid \text{No}) = 0\) se vuelve positiva, de modo que un día nublado ya no se asigna a Yes con certeza. El suavizado cuesta un poco de fidelidad a la muestra y compra robustez frente a los valores que la muestra, por azar, no contenía; con 14 días el efecto es visible, con 14,000 sería despreciable.

### 4.3 Bayes ingenuo gaussiano a mano

El ejemplo del clima usó características categóricas, para las cuales una verosimilitud es una frecuencia relativa. Para características continuas se necesita un modelo de densidad, y la variante gaussiana lo proporciona. Toma los seis casos de entrenamiento de la Unidad 1, con dos características \(x_1\) y \(x_2\), y el mismo caso nuevo \(\mathbf{x} = (15, 4)\) que el clasificador del centroide más cercano declaró benigno:

| caso | \(x_1\) | \(x_2\) | clase |
|---|---|---|---|
| 1 | 12 | 2 | benigno |
| 2 | 13 | 3 | benigno |
| 3 | 11 | 1 | benigno |
| 4 | 19 | 9 | maligno |
| 5 | 21 | 11 | maligno |
| 6 | 17 | 7 | maligno |

**Entrenamiento.** Las probabilidades a priori son \(P(B) = P(M) = 3/6 = 0.5\). Las medias por clase son \(\boldsymbol{\mu}_B = (12, 2)\) y \(\boldsymbol{\mu}_M = (19, 9)\). Usando la varianza <em>poblacional</em> (dividiendo por \(n_c\), como hace scikit-learn), las varianzas benignas son \(\sigma^2_{B1} = \tfrac{0 + 1 + 1}{3} = \tfrac{2}{3}\) y \(\sigma^2_{B2} = \tfrac{2}{3}\); las varianzas malignas son \(\sigma^2_{M1} = \tfrac{0 + 4 + 4}{3} = \tfrac{8}{3}\) y \(\sigma^2_{M2} = \tfrac{8}{3}\). La clase maligna está cuatro veces más dispersa que la benigna en ambas características.

**Clasificación** de \(\mathbf{x} = (15, 4)\): se usa la log-densidad \(\log \mathcal{N}(x \mid \mu, \sigma^2) = -\tfrac{1}{2}\log(2\pi\sigma^2) - \tfrac{(x-\mu)^2}{2\sigma^2}\). Sumar el logaritmo de la probabilidad a priori, \(\log 0.5\), a cada suma da \(\log P(B \mid \mathbf{x}) + \text{const} \approx -11.88\) y \(\log P(M \mid \mathbf{x}) + \text{const} \approx -11.20\). La clase maligna tiene la mayor log-probabilidad a posteriori, de modo que Bayes ingenuo clasifica el caso como **maligno**: lo contrario de la respuesta del centroide más cercano de la Unidad 1, aunque el caso está más cerca del centroide benigno. Normalizando, \(P(M \mid \mathbf{x}) = \tfrac{1}{1 + e^{-0.676}} \approx 0.663\). La razón es la varianza: un punto a 3 unidades del grupo benigno compacto es más sorprendente, bajo un modelo gaussiano, que un punto a 4–5 unidades del grupo maligno difuso. El bloque de código confirma la aritmética paso a paso.



In [ ]:
# Bayes ingenuo gaussiano a mano con los seis casos (varianza poblacional) y luego e1071 para comparar.
six <- rbind(c(12,2), c(13,3), c(11,1), c(19,9), c(21,11), c(17,7)); colnames(six) <- c("x1", "x2")
cls <- factor(c("benign","benign","benign","malignant","malignant","malignant"))
newx <- c(x1 = 15, x2 = 4)
log_gauss <- function(x, m, v) -0.5 * log(2 * pi * v) - (x - m)^2 / (2 * v)
logpost <- sapply(levels(cls), function(c) {
  Xc <- six[cls == c, , drop = FALSE]; m <- colMeans(Xc); v <- colMeans(sweep(Xc, 2, m)^2)   # varianza poblacional
  terms <- log_gauss(newx, m, v)
  cat(sprintf("%-9s mean (%.0f,%.0f) var (%.2f,%.2f)  log p(x_j|c)=(%.3f,%.3f)  log-prior+sum=%.3f\n",
              c, m[1], m[2], v[1], v[2], terms[1], terms[2], log(mean(cls == c)) + sum(terms)))
  log(mean(cls == c)) + sum(terms)
})
pM <- 1 / (1 + exp(logpost["benign"] - logpost["malignant"]))
cat(sprintf("by hand:  P(malignant | x) = %.4f,  P(benign | x) = %.4f\n", pM, 1 - pM))
nb6 <- naiveBayes(Class ~ ., data.frame(six, Class = cls))
cat(sprintf("e1071 naiveBayes (sample variance, n-1): P(malignant | x) = %.4f\n",
            predict(nb6, data.frame(t(newx)), type = "raw")[1, "malignant"]))



El cálculo a mano da \(P(M \mid \mathbf{x}) = 0.6629\), exactamente el resultado de Python. <code>e1071::naiveBayes</code> devuelve 0.497 —un casi empate que se inclina hacia benigno—, y la discrepancia es instructiva y vale la pena tenerla presente.

<div class="box warn"><h4>Advertencia: n frente a n − 1</h4><p><code>GaussianNB</code> de scikit-learn usa la varianza <em>poblacional</em> (divide la suma de desviaciones al cuadrado por \(n_c\)); <code>e1071::naiveBayes</code> usa la varianza <em>muestral</em> (divide por \(n_c - 1\)). Con solo tres casos por clase la diferencia es grande —ensancha cada gaussiana en un factor \(3/2\)— y basta para mover este punto limítrofe de maligno (0.66, poblacional) a un casi empate que se inclina hacia benigno (0.50, muestral). Con los cientos de casos por clase de los datos reales (Sección 4.4), el factor \(n_c/(n_c-1)\) es despreciable y las dos convenciones coinciden hasta el tercer decimal. La lección no es que una convención esté equivocada, sino que una probabilidad a posteriori situada sobre la frontera es sensible a pequeñas decisiones de modelado: justo la región de superposición donde ningún método puede tener confianza.</p></div>

### 4.4 Un clasificador de Bayes ingenuo desde cero, comparado con e1071

El cálculo con seis casos se generaliza sin cambios a \(n\) casos y \(p\) características. La función siguiente es un clasificador de Bayes ingenuo gaussiano completo en unas quince líneas: <code>fit</code> almacena los logaritmos de las probabilidades a priori, las medias y las varianzas; <code>predict</code> suma las log-densidades de cada característica y normaliza con el truco log-sum-exp. Se prueba con las nueve características de los datos de cáncer de mama frente a <code>e1071::naiveBayes</code>.



In [ ]:
# Un clasificador de Bayes ingenuo gaussiano en R base y una comparación con e1071 usando las nueve características.
hand_gnb_fit <- function(X, y, pop = TRUE) {
  cl <- levels(y); denom <- if (pop) 1 else NULL
  list(cl = cl, logprior = log(sapply(cl, function(c) mean(y == c))),
       mean = t(sapply(cl, function(c) colMeans(X[y == c, , drop = FALSE]))),
       var  = t(sapply(cl, function(c) { Xc <- X[y == c, , drop = FALSE]
                 colSums(sweep(Xc, 2, colMeans(Xc))^2) / (nrow(Xc) - !pop) })))
}
hand_gnb_logpost <- function(m, X) {
  ll <- sapply(seq_along(m$cl), function(k)
    rowSums(-0.5 * log(2 * pi * matrix(m$var[k, ], nrow(X), ncol(X), byrow = TRUE)) -
            sweep(X, 2, m$mean[k, ])^2 / (2 * matrix(m$var[k, ], nrow(X), ncol(X), byrow = TRUE))))
  joint <- sweep(ll, 2, m$logprior, "+")
  out <- joint - apply(joint, 1, function(r) log(sum(exp(r - max(r)))) + max(r))   # normaliza sobre las clases
  colnames(out) <- m$cl; out
}
Xall <- as.matrix(bc[, 1:9]); ytr <- bc$Class[idx]
Xtr <- as.matrix(tr[, 1:9]); Xte <- as.matrix(te[, 1:9])
hand <- hand_gnb_fit(Xtr, tr$Class, pop = TRUE)
P_hand <- exp(hand_gnb_logpost(hand, Xte))
nb9 <- naiveBayes(tr[, 1:9], tr$Class)
P_e1071 <- predict(nb9, Xte, type = "raw")[, levels(bc$Class)]
cat(sprintf("largest difference in any posterior, hand vs e1071: %.2e  (negligible: both use hundreds of cases per class)\n",
            max(abs(P_hand[, hand$cl] - P_e1071[, hand$cl]))))
pred_hand <- factor(hand$cl[max.col(hand_gnb_logpost(hand, Xte))], levels = levels(bc$Class))
cat(sprintf("test accuracy, 9 features: hand-written %.3f | e1071 %.3f\n",
            mean(pred_hand == te$Class), mean(predict(nb9, Xte) == te$Class)))
cat(sprintf("test accuracy, 2 features: e1071 %.3f\n", mean(predict(nb_p, te) == te$Class)))



Las dos implementaciones coinciden con una diferencia de a lo sumo 0.007 en cada probabilidad a posteriori —el pequeño residuo de la convención de varianza poblacional frente a muestral (Sección 4.3), despreciable con cientos de casos por clase— y alcanzan la misma exactitud en los casos reservados: por encima de 0.95 con las nueve características, y algo menor con las dos características de las figuras. El entrenamiento requirió una sola pasada sobre los datos para calcular nueve medias y nueve varianzas por clase: Bayes ingenuo es el clasificador más rápido de entrenar de este curso, lo cual es una de las razones por las que sigue siendo el modelo de referencia para problemas muy grandes.

La forma de la frontera se deduce del modelo. Para dos clases con características gaussianas, el logaritmo del cociente de probabilidades a posteriori es una función cuadrática de \(\mathbf{x}\) (los términos cuadráticos \((x_j - \mu_{cj})^2 / 2\sigma^2_{cj}\) tienen coeficientes distintos en las dos clases cuando las varianzas difieren), de modo que la frontera donde se anula es una sección cónica: una elipse, una parábola o una hipérbola. Si las varianzas fueran iguales en ambas clases, los términos cuadráticos se cancelarían y la frontera sería una recta: la misma forma lineal que la regresión logística, alcanzada por otro camino. El bloque de código siguiente dibuja las gaussianas condicionales de clase ajustadas y la frontera que producen en el problema de dos características.



In [ ]:
# El modelo gaussiano que ajusta Bayes ingenuo en dos dimensiones y la frontera (cuadrática) que implica.
shade_regions(function(G) predict(nb_p, G) == "malignant", xlim, ylim, cn = feat2, res = 150,
              main = "Gaussian naive Bayes: per-class Gaussians (1,2 sd) and boundary",
              xlab = "clump thickness", ylab = "cell size")
points(jitter(te[[feat2[1]]]), jitter(te[[feat2[2]]]), col = pcol(te), pch = 19, cex = 0.55)
th <- seq(0, 2 * pi, length.out = 100)
for (c in levels(tr$Class)) {
  m <- colMeans(tr[tr$Class == c, feat2]); s <- apply(tr[tr$Class == c, feat2], 2, sd)
  col <- if (c == "malignant") ORANGE else BLUE
  for (nsd in 1:2) lines(m[1] + nsd * s[1] * cos(th), m[2] + nsd * s[2] * sin(th), col = col, lwd = 1.3, lty = nsd)
  points(m[1], m[2], pch = 4, col = col, lwd = 3, cex = 1.6)
}
legend("topleft", c("malignant", "benign"), col = c(ORANGE, BLUE), pch = 19, bty = "n", cex = 0.8)



<p class="caption"><b>Figura 5.</b> El modelo detrás de la frontera: una gaussiana alineada con los ejes por clase (línea continua, 1 desviación estándar; discontinua, 2), con los centros marcados con cruces, y las regiones de decisión resultantes con los casos de prueba superpuestos. La clase maligna tiene una varianza mucho mayor en ambas características, por lo que su gaussiana es más ancha; la frontera se curva alrededor del grupo benigno compacto y es una curva, no una recta. Las elipses están alineadas con los ejes porque Bayes ingenuo supone que no hay correlación entre las dos características dentro de una clase —algo visiblemente falso para los casos malignos, que se distribuyen a lo largo de la diagonal—, y aun así la frontera es razonable.</p>

### 4.5 Generativo frente a discriminativo

Bayes ingenuo y la regresión logística (Unidad 2) pueden producir la misma familia de fronteras, y la comparación entre ambos es instructiva. La regresión logística es <em>discriminativa</em>: modela \(P(y \mid \mathbf{x})\) directamente y ajusta sus coeficientes para que las etiquetas de entrenamiento sean lo más probables posible, sin ningún modelo de cómo se distribuye \(\mathbf{x}\). Bayes ingenuo es <em>generativo</em>: modela \(p(\mathbf{x} \mid y)\) y \(P(y)\) y obtiene \(P(y \mid \mathbf{x})\) mediante el teorema de Bayes. Ng y Jordan mostraron, teóricamente y en muchos conjuntos de datos, que ambos tienen curvas de aprendizaje distintas: Bayes ingenuo se acerca a su error asintótico con muchos menos casos de entrenamiento, pero esa asíntota es más alta, porque sus supuestos lo sesgan; la regresión logística necesita más datos y termina siendo mejor.<sup class="c"><a href="#r7">[7]</a></sup> Con unas pocas decenas de casos, prefiere Bayes ingenuo; con miles, prefiere el modelo discriminativo. En los problemas médicos, donde los casos etiquetados son costosos, el régimen de muestras pequeñas es frecuente, y Bayes ingenuo tiene una larga trayectoria en el diagnóstico asistido por computador precisamente por esa razón.<sup class="c"><a href="#r8">[8]</a></sup>

### 4.6 Por qué Bayes ingenuo funciona aunque su supuesto sea falso

El enigma es empírico: Bayes ingenuo clasifica bien datos en los que las características son fuertemente dependientes. Domingos y Pazzani lo resolvieron en 1997.<sup class="c"><a href="#r5">[5]</a></sup> La clasificación con pérdida cero-uno depende solo de qué clase tiene la mayor probabilidad a posteriori, no del valor de esa probabilidad; el supuesto de independencia puede distorsionar considerablemente las probabilidades a posteriori sin cambiar su orden. Demostraron que Bayes ingenuo es óptimo para algunos problemas en los que el supuesto se viola groseramente, y mostraron en 28 conjuntos de datos que su exactitud no estaba relacionada con el grado de dependencia entre las características. Hand y Yu añadieron el argumento estadístico de que el modelo simple tiene pocos parámetros y, por lo tanto, baja varianza: su sesgo es real, pero con muestras pequeñas la varianza de un modelo más rico es peor.<sup class="c"><a href="#r6">[6]</a></sup> El precio del supuesto se paga en otra parte: las probabilidades a posteriori están mal <em>calibradas</em>, típicamente empujadas hacia 0 y 1, porque cada característica correlacionada cuenta como evidencia independiente. El bloque de código siguiente muestra ambas mitades de la historia.



In [ ]:
# Mitad 1: las características fuertemente dependientes no arruinan el orden (exactitud en CV de 5 pliegues con características redundantes).
cv_acc <- function(fitpred, X, y, K = 5) {
  set.seed(1); folds <- sample(rep(1:K, length.out = nrow(X)))
  mean(sapply(1:K, function(k) {
    tr_i <- folds != k; p <- fitpred(X[tr_i, , drop = FALSE], y[tr_i], X[!tr_i, , drop = FALSE])
    mean(p == y[!tr_i]) }))
}
nb_fp <- function(Xtr, ytr, Xte) predict(naiveBayes(Xtr, ytr), Xte)
size3 <- c("Cell.size", "Cell.shape", "Epith.c.size")                 # características de tamaño fuertemente correlacionadas
cat("Correlation among three size features (malignant class):\n")
print(round(cor(bc[bc$Class == "malignant", size3]), 3))
for (lbl in list(c("3 correlated size features", list(size3)), c("all 9 features", list(names(bc)[1:9]))))
  cat(sprintf("NB 5-fold CV accuracy with %-28s: %.3f\n", lbl[[1]], cv_acc(nb_fp, bc[, lbl[[2]], drop = FALSE], bc$Class)))
# Mitad 2: las probabilidades a posteriori son demasiado confiadas. Fracción de casos de prueba con una probabilidad a posteriori extrema, NB frente a regresión logística.
glm_fit <- suppressWarnings(glm(Class ~ ., data = tr, family = binomial))
p_nb  <- apply(predict(nb9, Xte, type = "raw"), 1, max)
p_lr  <- predict(glm_fit, te, type = "response"); p_lr <- pmax(p_lr, 1 - p_lr)
cat(sprintf("\nGaussian naive Bayes: accuracy %.3f | %.0f%% of test cases get posterior > 0.99 | mean max-posterior %.3f\n",
            mean(predict(nb9, Xte) == te$Class), 100 * mean(p_nb > 0.99), mean(p_nb)))
cat(sprintf("logistic regression : accuracy %.3f | %.0f%% of test cases get posterior > 0.99 | mean max-posterior %.3f\n",
            mean(ifelse(predict(glm_fit, te, type = "response") > 0.5, "malignant", "benign") == te$Class),
            100 * mean(p_lr > 0.99), mean(p_lr)))



El tamaño celular y la forma celular tienen una correlación superior a 0.7 dentro de una clase, y las tres características de tamaño están mutuamente asociadas (tabla impresa); tratarlas como independientes viola el supuesto, y sin embargo usar las nueve características —varias de ellas redundantes— no perjudica la exactitud en validación cruzada (0.96 frente a 0.95 con solo las tres características). La segunda mitad muestra el precio: Bayes ingenuo asigna una probabilidad a posteriori superior a 0.99 a una gran fracción de los casos de prueba, con más frecuencia que la regresión logística, porque cada característica redundante se cuenta como evidencia nueva. Cuando la salida de un clasificador va a usarse como probabilidad —para fijar un umbral sensible a los costos como en la Sección 3.2, o para mostrarla a un clínico—, las probabilidades a posteriori de Bayes ingenuo deben recalibrarse con datos reservados (el Suplemento S1 trata la calibración), o bien debe preferirse un modelo discriminativo.

<div class="box warn"><h4>Error frecuente</h4><p>"Bayes ingenuo es exacto, así que sus probabilidades son confiables". La exactitud depende del orden de las probabilidades a posteriori; la calibración depende de sus valores. Bayes ingenuo a menudo tiene lo primero sin lo segundo. Un 0.99 de un modelo de Bayes ingenuo con características correlacionadas no es un 99 % de probabilidad de malignidad; es una votación en la que el mismo testigo se contó varias veces.</p></div>

### 4.7 Bayes ingenuo para conteos y características binarias

La variante gaussiana es adecuada para mediciones continuas. Para datos de conteo —palabras en una nota clínica, lecturas mapeadas a un gen, píxeles de una imagen en escala de grises—, la variante multinomial modela cada clase mediante un vector de probabilidades de categoría estimadas como frecuencias suavizadas, y la variante de Bernoulli modela la presencia o ausencia de cada característica. Como R no tiene un conjunto de dígitos incorporado, el bloque de código siguiente <em>simula</em> imágenes de 8 × 8 de tres clases (una barra vertical, una barra horizontal y una diagonal, cada una con ruido), trata cada imagen como 64 conteos (multinomial) o como 64 características binarias de "tinta presente" (Bernoulli), y escribe ambas variantes desde cero; además ajusta la variante gaussiana para comparar. Lo importante no es la exactitud, sino que el mismo algoritmo sirve para tres tipos de datos cambiando solo el modelo unidimensional.



In [ ]:
# Imágenes simuladas de 8x8 similares a dígitos; Bayes ingenuo multinomial y de Bernoulli desde cero, más el gaussiano.
set.seed(0); npc <- 120; patterns <- list(vbar = NULL, hbar = NULL, diag = NULL)
base_imgs <- function(kind) {
  m <- matrix(1, 8, 8)                                                   # tinta tenue de fondo
  if (kind == "vbar") m[, 4:5] <- 8 else if (kind == "hbar") m[4:5, ] <- 8 else diag(m) <- 8
  as.vector(m)
}
Xdig <- do.call(rbind, lapply(names(patterns), function(k)
  t(replicate(npc, pmax(0, round(base_imgs(k) + rpois(64, 1) - 1))))))
ydig <- factor(rep(names(patterns), each = npc))
par(mfrow = c(1, 6), mar = c(0.2, 0.2, 1.4, 0.2))
for (i in c(1, 2, npc + 1, npc + 2, 2 * npc + 1, 2 * npc + 2))
  image(matrix(Xdig[i, ], 8)[, 8:1], col = grey.colors(12, 1, 0), axes = FALSE, main = as.character(ydig[i]))
# Bayes ingenuo de conteos/binario desde cero, con suavizado de Laplace
multinom_nb <- function(Xtr, ytr, Xte, binary = FALSE) {
  if (binary) { Xtr <- (Xtr > 3) + 0; Xte <- (Xte > 3) + 0 }
  cl <- levels(ytr); lp <- log(table(ytr) / length(ytr))
  if (binary) {
    theta <- sapply(cl, function(c) (colSums(Xtr[ytr == c, ]) + 1) / (sum(ytr == c) + 2))
    score <- sapply(cl, function(c) Xte %*% log(theta[, c]) + (1 - Xte) %*% log(1 - theta[, c]) + lp[c])
  } else {
    theta <- sapply(cl, function(c) (colSums(Xtr[ytr == c, ]) + 1) / (sum(Xtr[ytr == c, ]) + ncol(Xtr)))
    score <- sapply(cl, function(c) Xte %*% log(theta[, c]) + lp[c])
  }
  factor(cl[max.col(score)], levels = cl)
}
set.seed(2); ii <- sample(nrow(Xdig)); cut <- floor(0.7 * nrow(Xdig))
trI <- ii[1:cut]; teI <- ii[(cut + 1):nrow(Xdig)]
cat(sprintf("multinomial NB (counts)  test accuracy: %.3f\n", mean(multinom_nb(Xdig[trI, ], ydig[trI], Xdig[teI, ]) == ydig[teI])))
cat(sprintf("Bernoulli   NB (ink>3)   test accuracy: %.3f\n", mean(multinom_nb(Xdig[trI, ], ydig[trI], Xdig[teI, ], binary = TRUE) == ydig[teI])))
cat(sprintf("Gaussian    NB (e1071)   test accuracy: %.3f\n", mean(predict(naiveBayes(Xdig[trI, ], ydig[trI]), Xdig[teI, ]) == ydig[teI])))



<p class="caption"><b>Figura 5b.</b> Dos ejemplos simulados de cada una de las tres clases de imágenes (barra vertical, barra horizontal, diagonal), con ruido de Poisson añadido a cada píxel.</p>

Las variantes multinomial y de Bernoulli, escritas en unas pocas líneas, clasifican las tres clases simuladas de forma prácticamente perfecta con una sola pasada sobre los datos; la variante gaussiana obtiene un resultado comparable aquí porque las clases están bien separadas. Elegir la variante cuyo modelo unidimensional corresponde al tipo de característica es la única decisión de modelado que Bayes ingenuo le pide al analista.

<div class="exercise"><h4>Ejercicio 1</h4><p>En el bloque de código de la Sección 4.4, entrena los clasificadores escrito a mano y de <code>e1071</code> con solo los primeros 20 casos de entrenamiento (<code>tr[1:20, ]</code>) y compara su exactitud en prueba con la del modelo de regresión logística de la Sección 4.6 entrenado con los mismos 20 casos. Luego repite con 200 casos. ¿Qué modelo gana en cada régimen y cómo se relaciona esto con el resultado de Ng y Jordan?</p></div>

## 5. Redes bayesianas

### 5.1 De la distribución conjunta a un grafo

Bayes ingenuo hace un supuesto extremo —todas las características son independientes dada la clase— para que una distribución conjunta sea manejable. Una **red bayesiana** (red de creencias, modelo gráfico probabilístico) hace los supuestos explícitos y selectivos: es un grafo dirigido acíclico (DAG) cuyos nodos son variables aleatorias y cuyas flechas codifican dependencia directa, junto con, para cada nodo, una **tabla de probabilidad condicional (CPT)** que da la distribución del nodo para cada combinación de valores de sus padres. La semántica del grafo es una sola ecuación, la **factorización de la distribución conjunta**:

<div class="eq">\[ P(X_1, X_2, \dots, X_m) = \prod_{i=1}^{m} P\big(X_i \mid \mathrm{parents}(X_i)\big) \]
<span class="where">Cada variable depende directamente solo de sus padres en el grafo. La ecuación se deduce de la regla de la cadena de la probabilidad, \(P(X_1, \dots, X_m) = \prod_i P(X_i \mid X_1, \dots, X_{i-1})\), una vez que se supone que cada variable es independiente de sus no descendientes dados sus padres (la <em>propiedad de Markov local</em>).</span></div>

El formalismo fue desarrollado por Judea Pearl en la década de 1980 como representación del conocimiento incierto para sistemas expertos, reemplazando los factores de certeza ad hoc de MYCIN (Unidad 1) por una probabilidad coherente,<sup class="c"><a href="#r9">[9]</a>,<a href="#r10">[10]</a></sup> y el primer algoritmo general y eficiente de inferencia fue publicado por Lauritzen y Spiegelhalter en 1988 con un ejemplo médico que se ha convertido en la red estándar de enseñanza: una clínica torácica ficticia en la que una visita a Asia, el tabaquismo, la tuberculosis, el cáncer de pulmón y la bronquitis explican una radiografía de tórax y la disnea.<sup class="c"><a href="#r11">[11]</a></sup> Esta sección usa una versión reducida de esa red con cinco variables binarias —<em>Smoking</em> (S, tabaquismo), <em>Cancer</em> de pulmón (C), <em>Bronchitis</em> (B, bronquitis), <em>X-ray</em> de tórax positiva (X, radiografía) y <em>Dyspnoea</em> (D, disnea)—, conservando las probabilidades condicionales publicadas para las flechas que se mantienen. El bloque de código siguiente la dibuja e imprime las CPT.



In [ ]:
# La red 'Asia' reducida: un DAG con cinco variables binarias, dibujado con gráficos base; se imprimen las CPT.
pos <- list(S = c(0.5, 0.9), C = c(0.25, 0.55), B = c(0.75, 0.55), X = c(0.1, 0.15), D = c(0.6, 0.15))
lab <- list(S = "Smoking", C = "Lung\ncancer", B = "Bronchitis", X = "Positive\nX-ray", D = "Dyspnoea")
edges <- list(c("S","C"), c("S","B"), c("C","X"), c("C","D"), c("B","D"))
plot(NA, xlim = c(-0.05, 1.05), ylim = c(0, 1.02), axes = FALSE, xlab = "", ylab = "",
     main = "Bayesian network for a chest clinic (reduced Lauritzen-Spiegelhalter 'Asia')")
for (e in edges) { a <- pos[[e[1]]]; b <- pos[[e[2]]]; d <- b - a; u <- d / sqrt(sum(d^2))
  arrows(a[1] + 0.09 * u[1], a[2] + 0.09 * u[2], b[1] - 0.09 * u[1], b[2] - 0.09 * u[2], length = 0.12, col = GREY, lwd = 1.6) }
for (k in names(pos)) { p <- pos[[k]]
  symbols(p[1], p[2], circles = 0.085, inches = FALSE, add = TRUE, bg = ifelse(k %in% c("S","C","B"), "#dfe7f1", "#fbe3d3"), fg = BLUE)
  text(p[1], p[2], lab[[k]], cex = 0.8) }
text(1.0, 0.98, "blue: causes / risk factors\norange: observable findings", cex = 0.72, col = GREY, adj = 1)
cpt <- list(
  "P(S=yes)"        = c(`(no parents)` = 0.50),
  "P(C=yes | S)"    = c(`S=no` = 0.01, `S=yes` = 0.10),
  "P(B=yes | S)"    = c(`S=no` = 0.30, `S=yes` = 0.60),
  "P(X=yes | C)"    = c(`C=no` = 0.05, `C=yes` = 0.98),
  "P(D=yes | C,B)"  = c(`C=no,B=no` = 0.10, `C=no,B=yes` = 0.80, `C=yes,B=no` = 0.70, `C=yes,B=yes` = 0.90))
for (nm in names(cpt)) { cat(nm, ":\n"); print(cpt[[nm]]); cat("\n") }
cat(sprintf("free parameters in the factorised model: %d  (versus 2^5 - 1 = 31 for the unrestricted joint)\n",
            sum(sapply(cpt, length))))



<p class="caption"><b>Figura 6.</b> La red. El tabaquismo es un factor de riesgo tanto del cáncer de pulmón como de la bronquitis; el cáncer de pulmón produce una radiografía de tórax anormal y disnea; la bronquitis produce disnea, pero no una radiografía anormal. No hay flecha de Smoking a X-ray: el grafo afirma que, una vez que se conoce la presencia o ausencia de cáncer, el antecedente de tabaquismo no aporta información adicional sobre la radiografía. Cada flecha ausente de este tipo es un supuesto de independencia condicional.</p>

La factorización implicada por la Figura 6 es \(P(S, C, B, X, D) = P(S)\, P(C \mid S)\, P(B \mid S)\, P(X \mid C)\, P(D \mid C, B)\). Contar los parámetros muestra lo que aporta el grafo. La distribución conjunta completa de cinco variables binarias tiene \(2^5 - 1 = 31\) probabilidades libres; la forma factorizada necesita una para \(P(S)\), dos para cada una de \(P(C \mid S)\), \(P(B \mid S)\) y \(P(X \mid C)\), y cuatro para \(P(D \mid C, B)\): once en total, cada una de ellas una cantidad que un clínico puede enunciar o que un registro puede estimar.

### 5.2 Inferencia por enumeración, recorrida a mano

Hacer inferencia en una red bayesiana significa calcular la distribución a posteriori de algunas variables (la <em>consulta</em>) dados los valores observados de otras (la <em>evidencia</em>), con las variables restantes sin observar. El algoritmo exacto más simple es la **enumeración**: escribir la probabilidad conjunta de cada asignación completa usando la factorización, conservar las filas compatibles con la evidencia y sumar sobre las variables no observadas:

\[ P(C \mid D = \text{yes}, X = \text{yes}) = \frac{\sum_{S, B} P(S, C, B, X = \text{yes}, D = \text{yes})}{\sum_{S, C, B} P(S, C, B, X = \text{yes}, D = \text{yes})}. \]

**A mano, la probabilidad a priori del cáncer.** Antes de cualquier evidencia, \(P(C = \text{yes}) = \sum_S P(C = \text{yes} \mid S)\,P(S) = 0.10 \times 0.5 + 0.01 \times 0.5 = 0.055\). Así, en esta población el 5.5 % tiene cáncer de pulmón. **Con disnea observada**, la suma recorre las cuatro configuraciones ocultas de \((S, B)\) ponderadas por la conjunta factorizada; el numerador conserva \(C = \text{yes}\), el denominador ambos valores de \(C\), y el cociente resulta cercano a 0.10: un aumento modesto, porque la bronquitis explica igual de bien la disnea. **Añadir una radiografía positiva** multiplica cada fila superviviente por \(P(X = \text{yes} \mid C)\), que vale 0.98 para \(C = \text{yes}\) y solo 0.05 para \(C = \text{no}\); esto inclina fuertemente el cociente hacia el cáncer, hasta cerca de 0.69. El bloque de código construye la conjunta de 32 filas a partir de las CPT y una función general de consulta, y confirma cada número.



In [ ]:
# La distribución conjunta completa de la red (32 filas), construida a partir de las CPT, y una función de consulta por enumeración.
pS <- 0.50; pC <- c(0.01, 0.10); pB <- c(0.30, 0.60); pX <- c(0.05, 0.98)       # indexado por valor del padre + 1
pD <- c("0,0" = 0.10, "0,1" = 0.80, "1,0" = 0.70, "1,1" = 0.90)                 # con clave (C, B)
bern <- function(p, v) ifelse(v == 1, p, 1 - p)
jt <- expand.grid(S = 0:1, C = 0:1, B = 0:1, X = 0:1, D = 0:1)
jt$p <- with(jt, bern(pS, S) * bern(pC[S + 1], C) * bern(pB[S + 1], B) *
                 bern(pX[C + 1], X) * bern(pD[paste(C, B, sep = ",")], D))
cat(sprintf("rows: %d | probabilities sum to %.6f\n", nrow(jt), sum(jt$p)))
query <- function(target, ev = list()) {                                        # P(objetivo = yes | evidencia)
  sub <- jt; for (v in names(ev)) sub <- sub[sub[[v]] == ev[[v]], ]
  sum(sub$p[sub[[target]] == 1]) / sum(sub$p)
}
cat(sprintf("P(C = yes)                    = %.4f   (prior probability of lung cancer)\n", query("C")))
cat(sprintf("P(C = yes | D = yes)          = %.4f\n", query("C", list(D = 1))))
cat(sprintf("P(C = yes | D = yes, X = yes) = %.4f\n", query("C", list(D = 1, X = 1))))
print(head(jt[order(-jt$p), ], 4), row.names = FALSE)



La probabilidad a priori de cáncer de pulmón es 0.055, la disnea la eleva a cerca de 0.10 y una radiografía positiva adicional la eleva a cerca de 0.69, exactamente como predijo el cálculo a mano. Los estados completos más probables (tabla) son, como era de esperar, los sanos. El siguiente bloque de código añade evidencias una a una en un orden clínicamente natural y observa cómo se mueve la probabilidad a posteriori, incluido el efecto de saber que el paciente <em>no</em> fuma y de saber que hay bronquitis.



In [ ]:
# Cómo se mueve la probabilidad a posteriori de cáncer de pulmón a medida que se acumula evidencia (y aparecen explicaciones alternativas).
scen <- list("no evidence" = list(), "dyspnoea" = list(D = 1), "dyspnoea + positive X-ray" = list(D = 1, X = 1),
             "dyspnoea + X-ray + smoker" = list(D = 1, X = 1, S = 1), "dyspnoea + X-ray + non-smoker" = list(D = 1, X = 1, S = 0),
             "dyspnoea + bronchitis known" = list(D = 1, B = 1), "positive X-ray only" = list(X = 1),
             "negative X-ray + dyspnoea" = list(X = 0, D = 1))
pv <- sapply(scen, function(e) query("C", e))
par(mar = c(4, 13, 2, 1))
bp <- barplot(rev(pv), horiz = TRUE, las = 1, xlim = c(0, 1), col = ifelse(rev(pv) > 0.3, ORANGE, BLUE),
              xlab = "P(lung cancer = yes | evidence)", main = "Posterior probability of lung cancer under different evidence")
text(rev(pv) + 0.02, bp, sprintf("%.3f", rev(pv)), adj = 0, cex = 0.8)



<p class="caption"><b>Figura 7.</b> Probabilidad a posteriori de cáncer de pulmón bajo ocho conjuntos de evidencia. Cada elemento de evidencia mueve la probabilidad a posteriori a través de las flechas del grafo: la disnea y la radiografía son efectos del cáncer y la elevan; el tabaquismo es una causa y la eleva aún más; no fumar la reduce. Dos filas merecen atención. Saber que hay bronquitis <em>reduce</em> la probabilidad a posteriori de cáncer dada la disnea (compara "dyspnoea" y "dyspnoea + bronchitis known"): la bronquitis <em>explica</em> la disnea (efecto de explicación alternativa, explaining away), de modo que queda menos por explicar mediante el cáncer, aunque el cáncer y la bronquitis no estén conectados directamente. Y una radiografía negativa con disnea deja el cáncer como muy improbable, porque en esta red la radiografía tiene una sensibilidad de 0.98 para el cáncer.</p>

### 5.3 d-separación y aprendizaje de una red a partir de datos

Qué variables son independientes de cuáles, y dado qué, puede leerse en el grafo sin ninguna aritmética, mediante un criterio que Pearl llamó **d-separación**.<sup class="c"><a href="#r10">[10]</a></sup> Tres estructuras locales lo explican todo:

- **Cadena** \(S \rightarrow C \rightarrow X\): el tabaquismo influye en la radiografía solo a través del cáncer. \(S\) y \(X\) son dependientes, pero se vuelven independientes una vez que se observa \(C\): la cadena queda <em>bloqueada</em> al condicionar en el nodo intermedio.
- **Bifurcación** (causa común) \(C \leftarrow S \rightarrow B\): el cáncer y la bronquitis están asociados porque ambos son causados por el tabaquismo, y se vuelven independientes una vez que se conoce el tabaquismo. Condicionar en la causa común bloquea la bifurcación.
- **Colisionador** (efecto común) \(C \rightarrow D \leftarrow B\): el cáncer y la bronquitis son independientes (dado el tabaquismo) hasta que se observa el efecto común, la disnea; condicionar en un colisionador <em>abre</em> el camino y hace dependientes a las dos causas: el mecanismo de explicación alternativa (explaining away).

Dos variables están d-separadas, y por lo tanto son condicionalmente independientes, cuando todos los caminos entre ellas están bloqueados. El bloque de código verifica numéricamente las tres afirmaciones y luego muestra el **aprendizaje de parámetros** en su forma más simple: extrae de la red un registro sintético de 5,000 pacientes mediante muestreo hacia adelante (cada variable se muestrea después de sus padres) y recupera dos CPT contando.



In [ ]:
# Verificación numérica de las tres afirmaciones de d-separación y luego aprendizaje de parámetros por conteo.
cat(sprintf("Chain  S->C->X : P(X|C)=%.3f  P(X|C,S=yes)=%.3f  P(X|C,S=no)=%.3f  (equal: blocked by C)\n",
            query("X", list(C = 1)), query("X", list(C = 1, S = 1)), query("X", list(C = 1, S = 0))))
cat(sprintf("Fork   C<-S->B : P(B)=%.3f vs P(B|C=yes)=%.3f (dependent); given S: P(B|S)=%.3f vs P(B|S,C)=%.3f (independent)\n",
            query("B"), query("B", list(C = 1)), query("B", list(S = 1)), query("B", list(S = 1, C = 1))))
cat(sprintf("Collider C->D<-B: given S, P(C|S)=%.3f vs P(C|S,B)=%.3f (indep); given S,D: P(C|S,D)=%.3f vs P(C|S,D,B)=%.3f (dep)\n",
            query("C", list(S = 1)), query("C", list(S = 1, B = 1)), query("C", list(S = 1, D = 1)), query("C", list(S = 1, D = 1, B = 1))))
set.seed(0); N <- 5000
S <- runif(N) < pS; C <- runif(N) < ifelse(S, pC[2], pC[1]); B <- runif(N) < ifelse(S, pB[2], pB[1])
Xr <- runif(N) < ifelse(C, pX[2], pX[1]); D <- runif(N) < pD[paste(as.integer(C), as.integer(B), sep = ",")]
reg <- data.frame(S, C, B, D)
estC <- tapply(reg$C, reg$S, mean)
cat(sprintf("\nparameter learning by counting:  estimated P(C=yes | S=no)=%.3f (true 0.010), P(C=yes | S=yes)=%.3f (true 0.100)\n",
            estC["FALSE"], estC["TRUE"]))



Los números confirman el grafo: la probabilidad de radiografía positiva dado el cáncer es la misma fume o no el paciente (cadena bloqueada); la bronquitis está asociada con el cáncer marginalmente, pero no una vez fijado el tabaquismo (bifurcación bloqueada); y el cáncer y la bronquitis son independientes dado el tabaquismo hasta que se observa la disnea, momento en el cual saber que hay bronquitis reduce la probabilidad de cáncer (colisionador abierto). Las CPT estimadas siguen de cerca los valores verdaderos; su precisión depende del número de casos en cada configuración de los padres, de modo que un nodo con muchos padres necesita un registro grande o una distribución a priori: la lección general del aprendizaje de parámetros.<sup class="c"><a href="#r13">[13]</a></sup> Tanto los parámetros como la estructura pueden aprenderse a partir de datos: el <em>Bayes ingenuo aumentado con árbol</em> (tree-augmented naive Bayes) de Friedman, Geiger y Goldszmidt permite que cada característica tenga un padre adicional y mejora a Bayes ingenuo en la mayoría de los conjuntos de referencia (benchmarks),<sup class="c"><a href="#r14">[14]</a></sup> y la búsqueda de estructura en el espacio de los DAG, guiada por restricciones de expertos, subyace a las redes biomédicas que revisan Lucas y colaboradores.<sup class="c"><a href="#r15">[15]</a></sup> La importancia práctica del caso del colisionador es grande: seleccionar pacientes según un efecto común —el ingreso al hospital, un tamizaje positivo— induce asociaciones entre sus causas que no existen en la población, el sesgo que la epidemiología llama sesgo de colisionador o sesgo de Berkson.

<div class="box bridge"><h4>Puente con la Unidad 1 y con la bioinformática</h4><p>Una red bayesiana es la forma moderna del "razonamiento con incertidumbre" que la Unidad 1 atribuyó a la IA simbólica: el grafo es conocimiento, escrito a mano o aprendido, y la inferencia es deducción con probabilidades. El mismo formalismo subyace a los modelos ocultos de Markov que se usan para la búsqueda de genes y el alineamiento de perfiles en BGM324 —un HMM es una red bayesiana desplegada sobre las posiciones de una secuencia— y a los análisis de ligamiento genético en genealogías, donde el DAG es el árbol familiar.</p></div>

<div class="exercise"><h4>Ejercicio 2</h4><p>Añade el nodo que falta de la red Asia original: una visita a Asia (A, probabilidad a priori 0.01) eleva la probabilidad de tuberculosis (T: 0.05 si A, 0.01 en caso contrario), y tanto la tuberculosis como el cáncer producen la radiografía positiva y la disnea (reemplaza C por "T o C" en esas dos CPT). Reconstruye la tabla conjunta (ahora de 128 filas) y vuelve a calcular \(P(C \mid D = \text{yes}, X = \text{yes})\). ¿Por qué disminuye?</p></div>

## 6. k vecinos más cercanos

### 6.1 La regla, a mano

La regla de los **k vecinos más cercanos** (k-NN) es el clasificador basado en memoria más simple y el arquetipo de lo que la literatura llama un **aprendiz perezoso** (lazy learner): no construye ningún modelo explícito durante el entrenamiento —entrenar consiste en almacenar la muestra etiquetada— y posterga todo el cálculo hasta el momento en que se solicita una predicción. Para clasificar un caso nuevo \(\mathbf{x}\): calcula su distancia a cada caso almacenado, selecciona los \(k\) más cercanos y asigna la **clase mayoritaria** entre ellos; en regresión, predice la **media** de sus valores objetivo. La regla fue propuesta en un informe técnico de 1951 de Fix y Hodges, que permaneció inédito hasta 1989,<sup class="c"><a href="#r16">[16]</a></sup> y su propiedad teórica clave la establecieron Cover y Hart en 1967: a medida que la muestra de entrenamiento crece sin límite, el error de la regla de 1 vecino más cercano es a lo sumo el doble del error de Bayes, para cualquier distribución de los datos.<sup class="c"><a href="#r17">[17]</a></sup> La mitad de la información necesaria para una clasificación óptima está contenida en el único vecino más cercano: una afirmación notable para una regla sin modelo, sin entrenamiento y con un solo parámetro libre. La función que se usa aquí, <code>class::knn</code>, es la implementación de Venables y Ripley.<sup class="c"><a href="#r38">[38]</a></sup>

**Ejemplo resuelto.** Con los seis casos de entrenamiento de la Sección 4.3 y el caso nuevo \(\mathbf{x} = (15, 4)\), las distancias euclidianas son:

| caso | características | clase | \(d^2 = (15 - x_1)^2 + (4 - x_2)^2\) | \(d\) |
|---|---|---|---|---|
| 1 | (12, 2) | benigno | \(9 + 4 = 13\) | 3.61 |
| 2 | (13, 3) | benigno | \(4 + 1 = 5\) | 2.24 |
| 3 | (11, 1) | benigno | \(16 + 9 = 25\) | 5.00 |
| 4 | (19, 9) | maligno | \(16 + 25 = 41\) | 6.40 |
| 5 | (21, 11) | maligno | \(36 + 49 = 85\) | 9.22 |
| 6 | (17, 7) | maligno | \(4 + 9 = 13\) | 3.61 |

Para \(k = 1\) el caso más cercano es el caso 2 (benigno), de modo que la predicción es **benigno**. Para \(k = 3\) los tres más cercanos son el caso 2 (\(d = 2.24\)) y luego los casos 1 y 6, empatados en \(d = 3.61\); los tres vecinos incluyen dos casos benignos, y la mayoría vuelve a ser **benigno**. La variante ponderada por distancia de la Sección 6.6, en la que cada vecino vota con peso \(1/d\), da para benigno \(1/2.24 + 1/3.61 = 0.723\) frente a \(1/3.61 = 0.277\) para maligno: benigno por un margen más amplio. Los tres clasificadores de esta unidad discrepan ahora sobre este único punto: el centroide más cercano y k-NN dicen benigno, Bayes ingenuo gaussiano dice maligno, y el punto se encuentra en la superposición donde ningún método puede estar seguro. El bloque de código confirma la tabla.



In [ ]:
# k-NN a mano con los seis casos: distancias, los k más cercanos, la votación, confirmados con class::knn.
d <- sqrt(rowSums(sweep(six, 2, newx)^2))
tab <- data.frame(case = 1:6, x1 = six[, 1], x2 = six[, 2], class = as.character(cls), distance = round(d, 3))
print(tab[order(tab$distance), ], row.names = FALSE)
for (k in c(1, 3)) {
  o <- order(d)[1:k]; votes <- table(cls[o])
  cat(sprintf("k=%d: neighbours = cases %s, votes (benign,malignant)=(%d,%d), prediction = %s\n",
              k, paste(order(d)[1:k], collapse = ","), votes["benign"], votes["malignant"],
              as.character(knn(six, matrix(newx, 1), cls, k = k))))
}
o3 <- order(d)[1:3]; w <- 1 / d[o3]
cat(sprintf("k=3, weights 1/d: P(benign) = %.3f\n", sum(w[cls[o3] == "benign"]) / sum(w)))



<code>class::knn</code> reproduce la tabla y los votos. El voto ponderado da \(P(\text{benign}) = 0.723\). Los empates en la propia votación (posibles cuando \(k\) es par) los rompe <code>class::knn</code> al azar; usar un \(k\) impar en problemas binarios los evita. El voto depende de \(k\), y esa dependencia puede invertir la respuesta. Las diapositivas de clase lo ilustran con una configuración que el bloque de código reproduce: un punto nuevo cuyos tres vecinos más cercanos son dos triángulos y un cuadrado, y cuyos cinco más cercanos son dos triángulos y tres cuadrados. Con \(k = 3\) el punto es un triángulo; con \(k = 5\), un cuadrado. Nada cambió en los datos: solo el tamaño del vecindario consultado.



In [ ]:
# El clásico ejemplo de juguete: el mismo punto nuevo es un 'triángulo' con k=3 y un 'cuadrado' con k=5.
toy_X <- rbind(c(0.6,0.8), c(-0.7,0.7), c(1.0,-0.5), c(-1.2,-1.0), c(1.3,1.1),
               c(-2.4,1.8), c(2.6,-1.9), c(-2.0,-2.5), c(2.8,2.4), c(0.3,3.0))
toy_y <- factor(c("tri","tri","sq","sq","sq","tri","sq","tri","sq","tri"))
dq <- sqrt(rowSums(toy_X^2)); ord <- order(dq)
par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
for (k in c(3, 5)) {
  v <- table(toy_y[ord[1:k]]); pred <- names(which.max(v))
  cat(sprintf("k = %d: votes triangles %d / squares %d  ->  %s\n", k, v["tri"], v["sq"], pred))
  plot(toy_X, type = "n", xlim = c(-3.2, 3.4), ylim = c(-3.2, 3.4), asp = 1, xlab = "", ylab = "",
       main = sprintf("k = %d: %d triangles, %d squares -> %s", k, v["tri"], v["sq"], pred))
  symbols(0, 0, circles = dq[ord[k]] * 1.02, inches = FALSE, add = TRUE, fg = GREY, lty = 2)
  points(toy_X[toy_y == "tri", ], pch = 17, col = BLUE, cex = 1.6)
  points(toy_X[toy_y == "sq", ], pch = 15, col = ORANGE, cex = 1.4)
  points(0, 0, pch = 8, col = GREEN, cex = 2, lwd = 2)
}



<p class="caption"><b>Figura 8.</b> Los mismos diez casos almacenados y el mismo punto nuevo (estrella), con el vecindario de los \(k\) casos más cercanos dibujado como un círculo discontinuo. Con \(k = 3\) el círculo contiene dos triángulos y un cuadrado, y el punto se clasifica como triángulo; con \(k = 5\) contiene dos triángulos y tres cuadrados, y el punto pasa a ser un cuadrado. La inversión no es un defecto de la regla, sino un recordatorio de que \(k\) es una decisión de modelado: la Sección 6.4 la toma mediante validación y no por inspección.</p>

### 6.2 Qué significa "más cercano": métricas de distancia

La regla depende por completo de la definición de distancia. Para dos vectores de características \(\mathbf{a}\) y \(\mathbf{b}\) en \(\mathbb{R}^p\), la **distancia de Minkowski** de orden \(q\) es

\[ d_q(\mathbf{a}, \mathbf{b}) = \Big( \sum_{j=1}^{p} |a_j - b_j|^q \Big)^{1/q}, \]

de la cual la distancia **euclidiana** (\(q = 2\)) y la distancia **Manhattan** o de cuadras (city-block) (\(q = 1\)) son los casos habituales; cuando \(q \to \infty\), la distancia tiende a la mayor diferencia en una sola coordenada (Chebyshev). La distancia euclidiana es la opción por defecto y es adecuada para características continuas en escalas coherentes; la Manhattan está menos dominada por una sola desviación grande y suele preferirse en dimensiones altas.<sup class="c"><a href="#r21">[21]</a></sup> Para características binarias, el análogo es la distancia de Hamming (número de discordancias), y para perfiles de expresión, la distancia coseno o de correlación, que ignora la magnitud global. El bloque de código calcula las distancias entre dos casos reales y dibuja el conjunto de puntos a distancia unitaria del origen bajo cada métrica: la forma que adopta un vecindario.



In [ ]:
# Tres distancias de Minkowski entre los mismos dos casos y el 'círculo' unitario de cada métrica.
a <- as.numeric(tr[1, feat2]); b <- as.numeric(tr[2, feat2])
cat(sprintf("case A = (%.0f,%.0f), case B = (%.0f,%.0f)\n", a[1], a[2], b[1], b[2]))
for (q in list(c(1, "Manhattan"), c(2, "Euclidean"), c(3, "Minkowski q=3"), c(Inf, "Chebyshev"))) {
  qq <- as.numeric(q[1]); dd <- if (is.infinite(qq)) max(abs(a - b)) else sum(abs(a - b)^qq)^(1 / qq)
  cat(sprintf("  %-14s distance = %.3f\n", q[2], dd))
}
th <- seq(0, 2 * pi, length.out = 400); u <- cbind(cos(th), sin(th))
plot(NA, xlim = c(-1.4, 1.4), ylim = c(-1.4, 1.4), asp = 1, xlab = "a1 - b1", ylab = "a2 - b2",
     main = "Points at distance 1 from the origin")
for (q in list(c(1, ORANGE), c(2, BLUE), c(Inf, GREEN))) {
  qq <- as.numeric(q[1])
  nrm <- if (is.infinite(qq)) apply(abs(u), 1, max) else rowSums(abs(u)^qq)^(1 / qq)
  lines(u / nrm, col = q[2], lwd = 2)
}
legend("topright", c("Manhattan (q=1)", "Euclidean (q=2)", "Chebyshev (q->inf)"),
       col = c(ORANGE, BLUE, GREEN), lwd = 2, bty = "n", cex = 0.8)



<p class="caption"><b>Figura 9.</b> El conjunto de puntos a distancia unitaria del centro bajo tres métricas: un rombo para Manhattan, un círculo para la euclidiana y un cuadrado para Chebyshev. Un vecindario de k-NN tiene la forma de la bola unitaria de la métrica, de modo que la elección de la métrica decide qué casos cuentan como vecinos. Las distancias impresas entre dos casos reales se ordenan de la forma esperada, \(d_1 \geq d_2 \geq d_\infty\).</p>

### 6.3 El escalado de las características no es opcional

Toda distancia de Minkowski suma diferencias entre características como si fueran conmensurables. En los datos de cáncer de mama de <code>mlbench</code> lo son —las nueve características son puntuaciones enteras de 1 a 10—, de modo que estandarizarlas cambia poco, y este es uno de los puntos en que el conjunto de datos de R es más benévolo que el de scikit-learn, cuyas características abarcan desde "mean area" en los cientos hasta "fractal dimension" en el segundo decimal. Para mostrar la lección que importa en general, el bloque de código <em>simula</em> una característica mal escalada multiplicando una característica (Mitoses) por 100, como si se hubiera registrado en otras unidades, y muestra que k-NN con las características sin escalar se convierte entonces en un clasificador de una sola característica, lo que la estandarización repara.



In [ ]:
# Escalado de características: con los datos originales (comparables) apenas importa; con una copia mal escalada es decisivo.
knn_cv <- function(X, y, k = 5, Kf = 5, standardize = FALSE) {
  set.seed(1); folds <- sample(rep(1:Kf, length.out = nrow(X)))
  mean(sapply(1:Kf, function(f) {
    trI <- folds != f; Xtr <- X[trI, , drop = FALSE]; Xte <- X[!trI, , drop = FALSE]
    if (standardize) { ct <- colMeans(Xtr); s <- apply(Xtr, 2, sd); Xtr <- scale(Xtr, ct, s); Xte <- scale(Xte, ct, s) }
    mean(knn(Xtr, Xte, y[trI], k = k) == y[!trI]) }))
}
Xnat <- as.matrix(bc[, 1:9]); set.seed(3)
Xaug <- cbind(Xnat, noise = rnorm(nrow(Xnat)) * 500)        # una característica registrada en una escala grande (aquí, ruido puro)
cat(sprintf("native features (all 1-10):         raw %.3f  |  standardised %.3f\n",
            knn_cv(Xnat, bc$Class), knn_cv(Xnat, bc$Class, standardize = TRUE)))
cat(sprintf("plus one large-scale noise feature:  raw %.3f  |  standardised %.3f\n",
            knn_cv(Xaug, bc$Class), knn_cv(Xaug, bc$Class, standardize = TRUE)))



Con los datos originales, donde cada característica ya está en la escala de 1–10, estandarizar apenas mueve la exactitud en validación cruzada. Si se añade una sola característica en una escala grande —aquí ruido puro multiplicado por 500—, k-NN sin escalar se desploma (a cerca de 0.78), porque la distancia euclidiana queda dominada por esa única característica y las nueve informativas se vuelven numéricamente invisibles; estandarizar lleva cada característica a varianza unitaria y restablece la exactitud. Lo mismo se aplica a la máquina de vectores de soporte de la Sección 7 y a todo método basado en distancias o productos internos; Bayes ingenuo y los árboles de decisión (Unidad 4) son las excepciones, ya que tratan cada característica en su propia escala.

<div class="box warn"><h4>Error frecuente</h4><p>"Estandarizar todo el conjunto de datos antes de dividirlo es inofensivo". La media y la desviación estándar de todo el conjunto de datos incluyen los casos de prueba, de modo que estos se han filtrado en el procedimiento de entrenamiento. La fuga es pequeña en el escalado y grande en otros pasos de preprocesamiento (selección de características, imputación); el hábito de ajustar el escalador solo con el pliegue de entrenamiento —como hace la función auxiliar <code>knn_cv</code>— la evita de manera uniforme. El Suplemento S1 trata la fuga de datos en detalle.</p></div>

### 6.4 Elección de k

El parámetro \(k\) controla la suavidad de la regla. Con \(k = 1\) cada caso de entrenamiento se clasifica correctamente —su vecino más cercano es él mismo— y la frontera rodea cada caso aislado, incluidos los mal etiquetados: la regla ha memorizado la muestra, lo que es sobreajuste en su forma más pura. A medida que \(k\) crece, la votación promedia sobre un vecindario más grande, la frontera se suaviza y la influencia de cualquier caso individual se desvanece; en el límite \(k = n\), a todo punto se le asigna la clase mayoritaria de toda la muestra. El \(k\) adecuado equilibra estos efectos y se elige mediante validación, nunca por el error de entrenamiento. El bloque de código calcula una **curva de validación**: la exactitud de entrenamiento y la exactitud en validación cruzada de 5 pliegues en función de \(k\), con las nueve características estandarizadas.



In [ ]:
# Curva de validación para k: exactitud de entrenamiento frente a exactitud en validación cruzada (nueve características estandarizadas).
Xs <- scale(Xnat); ks <- seq(1, 59, by = 2)
tr_acc <- sapply(ks, function(k) mean(knn(Xs, Xs, bc$Class, k = k) == bc$Class))
cv_a   <- sapply(ks, function(k) knn_cv(Xnat, bc$Class, k = k, standardize = TRUE))
best_k <- ks[which.max(cv_a)]
cat(sprintf("k=1: training accuracy %.3f, CV accuracy %.3f\n", tr_acc[1], cv_a[1]))
cat(sprintf("best k by CV: %d (CV accuracy %.3f); CV accuracy at k=59: %.3f\n", best_k, max(cv_a), cv_a[length(ks)]))
plot(ks, tr_acc, type = "b", pch = 19, cex = 0.5, col = GREY, ylim = c(0.93, 1.002),
     xlab = "k (number of neighbours)", ylab = "accuracy", main = "Validation curve of k-NN on the breast-cancer data")
lines(ks, cv_a, type = "b", pch = 19, cex = 0.5, col = BLUE)
abline(v = best_k, col = ORANGE, lty = 2)
legend("bottomright", c("training accuracy", "5-fold CV accuracy", sprintf("best k = %d", best_k)),
       col = c(GREY, BLUE, ORANGE), lty = c(1, 1, 2), pch = c(19, 19, NA), bty = "n", cex = 0.8)



<p class="caption"><b>Figura 10.</b> Exactitud de entrenamiento (gris) y exactitud en validación cruzada (azul) a medida que \(k\) va de 1 a 59. Con \(k = 1\) la exactitud de entrenamiento es 1.000 por construcción, mientras que la exactitud en validación cruzada es menor; la brecha es el sobreajuste. La curva de validación cruzada sube hasta un máximo amplio y luego desciende lentamente a medida que el vecindario se vuelve demasiado grande para seguir la frontera. El máximo es amplio: un rango de valores de \(k\) es defendible, y el único "mejor" valor no debe sobreinterpretarse (el Suplemento S1 trata la regla de un error estándar).</p>

El efecto de \(k\) sobre la forma de la frontera se ve mejor en dos dimensiones. El siguiente bloque de código ajusta k-NN con \(k = 1\), 5 y 25 al problema estandarizado de dos características y dibuja las regiones.



In [ ]:
# Regiones de decisión de k-NN para k = 1, 5 y 25 en el problema de dos características (estandarizado).
par(mfrow = c(1, 3), mar = c(4, 4, 3, 1))
for (k in c(1, 5, 25)) {
  predk <- function(G) knn(sc(tr), scale(G, center = ctr, scale = csd), tr$Class, k = k) == "malignant"
  tra <- mean(knn(sc(tr), sc(tr), tr$Class, k = k) == tr$Class); tea <- mean(knn(sc(tr), sc(te), tr$Class, k = k) == te$Class)
  shade_regions(predk, xlim, ylim, cn = feat2, res = 120,
                main = sprintf("k = %d: train %.3f, test %.3f", k, tra, tea),
                xlab = "clump thickness", ylab = if (k == 1) "cell size" else "")
  points(jitter(tr[[feat2[1]]]), jitter(tr[[feat2[2]]]), col = pcol(tr), pch = 19, cex = 0.4)
}



<p class="caption"><b>Figura 11.</b> Regiones de decisión para \(k = 1\), 5 y 25, con los casos de entrenamiento. Con \(k = 1\) la frontera es irregular, con islas alrededor de cada caso aislado (exactitud de entrenamiento 1.000, y las islas son ruido); con \(k = 5\) desaparecen la mayoría de las islas; con \(k = 25\) la frontera es suave y los pocos errores de entrenamiento que quedan son casos ubicados en lo profundo del territorio de la otra clase. Las exactitudes de prueba, impresas en los títulos, son similares para \(k = 5\) y 25, y menores para \(k = 1\).</p>

### 6.5 La maldición de la dimensionalidad

La regla k-NN supone que existen vecinos cercanos. En espacios de alta dimensión puede que no existan, en un sentido que Richard Bellman llamó <em>maldición de la dimensionalidad</em> en 1961<sup class="c"><a href="#r20">[20]</a></sup> y que Beyer y colaboradores precisaron en 1999: a medida que crece la dimensión \(p\), la distancia de un punto de consulta a su vecino más cercano se aproxima a la distancia a su vecino más lejano, de modo que el contraste entre "cerca" y "lejos" desaparece.<sup class="c"><a href="#r21">[21]</a></sup> Para encerrar el 10 % de puntos distribuidos uniformemente en el cubo unitario se necesita un cubo de lado \(0.1^{1/p}\): 0.32 para \(p = 2\), 0.63 para \(p = 5\), 0.79 para \(p = 10\); el "vecindario" abarca la mayor parte de cada eje. El bloque de código simula la concentración de las distancias.



In [ ]:
# Concentración de distancias: cociente entre la distancia al vecino más lejano y al más cercano a medida que crece la dimensión (datos uniformes).
set.seed(0); dims <- c(1, 2, 3, 5, 10, 20, 50, 100, 200, 500, 1000); npts <- 500; nq <- 50
ratio <- sapply(dims, function(p) {
  pts <- matrix(runif(npts * p), npts); qs <- matrix(runif(nq * p), nq)
  median(apply(qs, 1, function(q) { dd <- sqrt(colSums((t(pts) - q)^2)); max(dd) / min(dd) })) })
summ <- data.frame(p = dims, farthest_over_nearest = round(ratio, 2), side_for_10pct = round(0.1^(1 / dims), 2))
print(summ, row.names = FALSE)
plot(dims, ratio, type = "b", pch = 19, col = BLUE, log = "xy", xlab = "number of dimensions p",
     ylab = "farthest / nearest (median)", main = "Distance concentration: 'nearest' loses meaning as p grows")



<p class="caption"><b>Figura 12.</b> Mediana del cociente entre la distancia al punto más lejano y al más cercano entre 500 puntos uniformes, en función de la dimensión. En una dimensión, el punto más lejano está cientos de veces más lejos que el más cercano; en \(p = 1000\) está apenas un 10 % más lejos (tabla impresa). La última columna da el lado del cubo necesario para encerrar el 10 % de los datos. Los datos reales se ubican cerca de estructuras de menor dimensión, y por eso k-NN funciona siquiera con nueve o sesenta y cuatro características; pero con miles de características (expresión génica, espectros) k-NN necesita selección de características o una distancia aprendida.</p>

### 6.6 k-NN ponderado, regresión y costo

Dos extensiones son habituales. La **ponderación por distancia** permite que cada vecino vote con un peso que disminuye con la distancia, típicamente \(1/d\), de modo que un vecino cercano cuenta más; Dudani la introdujo en 1976 y reduce la sensibilidad al valor exacto de \(k\).<sup class="c"><a href="#r18">[18]</a></sup> La **regresión k-NN** predice una variable objetivo continua mediante la media (ponderada) de los valores objetivo de los vecinos; es la regresión no paramétrica más simple, y la exposición de Altman es la introducción estándar.<sup class="c"><a href="#r19">[19]</a></sup> Como R no tiene un conjunto de datos de diabetes incorporado, el bloque de código aplica ambas a <code>BostonHousing</code> —506 suburbios, 13 predictores y un valor mediano continuo de la vivienda, el sustituto en R de la regresión con datos de diabetes del curso en Python<sup class="c"><a href="#r35">[35]</a></sup>— y compara el \(R^2\) en validación cruzada con el de la regresión lineal; además mide el tiempo de la predicción frente al del "entrenamiento" para mostrar dónde invierte su esfuerzo k-NN.



In [ ]:
# Regresión k-NN con BostonHousing (pesos uniformes y por distancia) frente a regresión lineal; y el perfil de costo.
data(BostonHousing, package = "mlbench")
Xb <- scale(data.matrix(BostonHousing[, 1:13])); yb <- BostonHousing$medv   # data.matrix: factor chas -> numérico
knn_reg <- function(Xtr, Xte, ytr, k, weights = "uniform") {
  D <- as.matrix(dist(rbind(Xte, Xtr)))[1:nrow(Xte), -(1:nrow(Xte))]
  apply(D, 1, function(dr) { o <- order(dr)[1:k]; w <- if (weights == "distance") 1 / pmax(dr[o], 1e-9) else rep(1, k)
                             sum(w * ytr[o]) / sum(w) }) }
set.seed(1); folds <- sample(rep(1:5, length.out = nrow(Xb)))
cvR2 <- function(k, w) { pr <- numeric(nrow(Xb))
  for (f in 1:5) { trI <- folds != f; pr[!trI] <- knn_reg(Xb[trI, ], Xb[!trI, ], yb[trI], k, w) }
  1 - sum((yb - pr)^2) / sum((yb - mean(yb))^2) }
res <- sapply(c("uniform", "distance"), function(w) sapply(c(1, 5, 15, 30), function(k) round(cvR2(k, w), 3)))
rownames(res) <- paste0("k=", c(1, 5, 15, 30)); print(res)
linR2 <- { pr <- numeric(nrow(Xb)); df <- BostonHousing
  for (f in 1:5) { trI <- folds != f; pr[!trI] <- predict(lm(medv ~ ., df[trI, ]), df[!trI, ]) }
  1 - sum((yb - pr)^2) / sum((yb - mean(yb))^2) }
cat(sprintf("linear regression (Unit 2 method): CV R2 = %.3f\n", linR2))
big <- scale(matrix(rnorm(4000 * 20), 4000)); ybig <- factor(sample(c("a", "b"), 4000, TRUE))
t_fit <- system.time(invisible(NULL))["elapsed"]                                   # 'entrenamiento' de k-NN = almacenar: gratis
t_pred <- system.time(knn(big, big[1:800, ], ybig, k = 5))["elapsed"]
cat(sprintf("cost: storing 4000x20 training cases is free; predicting 800 queries took %.3f s (all distances computed)\n", t_pred))



El patrón repite el de la clasificación: un \(k\) intermedio es el mejor (alrededor de \(k = 5\)), un \(k\) muy grande suaviza hacia la media global, y la ponderación por distancia es lo que más ayuda: eleva el mejor \(R^2\) en validación cruzada a cerca de 0.79. Aquí k-NN <em>supera por poco</em> al modelo lineal (\(R^2 \approx 0.72\)), lo contrario del resultado con los datos de diabetes del curso en Python: el valor mediano de las viviendas depende de forma no lineal de los predictores, y en esos datos un método que no supone nada puede superar a uno que supone linealidad, siempre que la muestra sea lo bastante densa para encontrar vecinos cercanos. La lección es que qué familia gana es una propiedad de los datos, no del método. La medición de tiempos muestra el perfil de costo invertido de k-NN —el entrenamiento es gratis, y cada predicción calcula las \(n\) distancias—, al revés que todos los demás clasificadores de este curso. Las estructuras de partición del espacio (el árbol k-d de Bentley,<sup class="c"><a href="#r22">[22]</a> el ball tree) aceleran la búsqueda en dimensiones bajas, pero dejan de ayudar más allá de unas pocas decenas, una consecuencia de la Sección 6.5. Los métodos basados en memoria siguen siendo valiosos en biomedicina cuando la muestra es en sí misma el objeto de interés: la imputación k-NN de valores faltantes en matrices de expresión es un paso de preprocesamiento estándar.<sup class="c"><a href="#r23">[23]</a></sup>

<div class="exercise"><h4>Ejercicio 3</h4><p>En el bloque de código de la curva de validación de la Sección 6.4, cambia la distancia a Manhattan reemplazando el paso euclidiano de <code>knn</code> por un k-NN Manhattan explícito (o escala las características de otra manera). ¿Cambia el mejor \(k\)? Luego añade ponderación por distancia al ejemplo de los seis casos y explica por qué la exactitud de entrenamiento de un 1-NN ponderado por distancia debe seguir siendo 1.000.</p></div>

## 7. Máquinas de vectores de soporte

### 7.1 La idea del margen máximo

Volvamos a la visión geométrica. Una máquina de vectores de soporte es un <strong>clasificador discriminativo que entrega un hiperplano óptimo</strong>: dados casos de entrenamiento etiquetados, devuelve el hiperplano que mejor separa las clases, y los casos nuevos se clasifican según el lado en que caen. Los datos son **linealmente separables** cuando existe un hiperplano así —una recta en dos dimensiones, un plano en tres— y **no linealmente separables** cuando se requiere una frontera curva (Sección 7.3). Cuando dos clases son linealmente separables, infinitos hiperplanos las separan, y el perceptrón de la Unidad 1 se detiene en el primero que encuentra. Vapnik y sus colaboradores se preguntaron qué hiperplano separador es el <em>mejor</em>, y respondieron: el más alejado del caso de entrenamiento más cercano de cualquiera de las clases, es decir, el hiperplano con el **margen máximo**, la distancia perpendicular desde el hiperplano hasta los puntos de entrenamiento más cercanos.<sup class="c"><a href="#r24">[24]</a>,<a href="#r26">[26]</a></sup> El argumento es de robustez: una frontera que pasa cerca de los casos de entrenamiento clasificará mal casos nuevos que caigan ligeramente más allá de ellos, mientras que un corredor vacío ancho tolera esas perturbaciones. Los casos que se encuentran exactamente en el borde del corredor lo determinan y se llaman **vectores de soporte**; mover cualquier otro caso no cambia nada.

Escribe el hiperplano como \(\mathbf{w}^\top \mathbf{x} + b = 0\) y codifica las clases como \(y_i \in \{-1, +1\}\). Escalar \(\mathbf{w}\) y \(b\) de modo que los puntos más cercanos cumplan \(|\mathbf{w}^\top \mathbf{x}_i + b| = 1\) hace que el ancho del corredor sea \(2 / \lVert \mathbf{w} \rVert\), de modo que maximizar el margen equivale a minimizar \(\lVert \mathbf{w} \rVert\):

<div class="eq">\[ \min_{\mathbf{w}, b} \; \tfrac{1}{2} \lVert \mathbf{w} \rVert^2 \qquad \text{subject to} \qquad y_i (\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 \quad \text{for all } i. \]
<span class="where">Esta es la máquina de vectores de soporte de <strong>margen duro</strong>: un programa cuadrático convexo con solución única siempre que los datos sean separables. En la solución, \(\mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i\), con \(\alpha_i > 0\) solo para los vectores de soporte.</span></div>

La función de R <code>e1071::svm</code> es una interfaz a LIBSVM.<sup class="c"><a href="#r33">[33]</a>,<a href="#r37">[37]</a></sup> El bloque de código siguiente genera dos grupos separables, ajusta una máquina de margen duro (un \(C\) muy grande) y dibuja el hiperplano, los márgenes y los vectores de soporte.



In [ ]:
# El hiperplano de margen máximo con datos separables: hiperplano, corredor del margen y vectores de soporte.
bl <- make_blobs(60, rbind(c(0, 0), c(3.5, 3.5)), sd = 0.6); Xbl <- bl$X; ybl <- bl$y
hard <- svm(Xbl, ybl, kernel = "linear", cost = 1e5, scale = FALSE)
w <- as.numeric(t(hard$coefs) %*% hard$SV); b <- -hard$rho
cat(sprintf("w = (%.3f, %.3f), b = %.3f, ||w|| = %.3f, margin width 2/||w|| = %.3f\n", w[1], w[2], b, sqrt(sum(w^2)), 2 / sqrt(sum(w^2))))
cat(sprintf("support vectors: %d of %d cases\n", hard$tot.nSV, length(ybl)))
plot(Xbl, col = ifelse(ybl == levels(ybl)[1], ORANGE, BLUE), pch = 19, cex = 0.9,
     xlab = "x1", ylab = "x2", main = "Maximum-margin hyperplane (solid) and margins (dashed)")
abline(a = -b / w[2], b = -w[1] / w[2], lwd = 2)                               # w.x + b = 0
abline(a = (-b + 1) / w[2], b = -w[1] / w[2], lty = 2); abline(a = (-b - 1) / w[2], b = -w[1] / w[2], lty = 2)
points(hard$SV, cex = 2.2, col = GREEN, lwd = 2)
legend("topleft", c("class -1", "class +1", "support vectors"), col = c(ORANGE, BLUE, GREEN),
       pch = c(19, 19, 1), bty = "n", cex = 0.8)



<p class="caption"><b>Figura 13.</b> Sesenta puntos separables, el hiperplano de margen máximo (línea continua) y las dos líneas de margen (discontinuas) a distancia con signo ± 1. Los puntos encerrados en círculos son los vectores de soporte: se encuentran sobre las líneas de margen, y el hiperplano queda determinado solo por ellos. Cualquier otro punto podría moverse a cualquier lugar de su propio lado del corredor sin cambiar la solución: la solución es <em>dispersa</em> (sparse) en los datos, la propiedad que da nombre al método.</p>

### 7.2 El margen blando y el parámetro C

Las clases reales se superponen, y entonces el problema de margen duro no tiene solución. La formulación de Cortes y Vapnik de 1995 introdujo **variables de holgura** (slack) \(\xi_i \geq 0\), que miden cuánto viola cada caso el margen, y un parámetro \(C\) que pone precio a las violaciones:<sup class="c"><a href="#r25">[25]</a></sup>

<div class="eq">\[ \min_{\mathbf{w}, b, \boldsymbol{\xi}} \; \tfrac{1}{2} \lVert \mathbf{w} \rVert^2 + C \sum_{i} \xi_i \qquad \text{subject to} \qquad y_i (\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0. \]
<span class="where">Un <strong>\(C\) grande</strong> castiga cada violación: se clasifican mal menos casos de entrenamiento, el margen se reduce y la frontera se curva hacia los datos, con riesgo de <em>sobreajuste</em> (sesgo bajo, varianza alta). Un <strong>\(C\) pequeño</strong> tolera violaciones a cambio de un margen mayor: más general y más estable, con riesgo de <em>subajuste</em> (sesgo alto, varianza baja).</span></div>

\(C\) es el parámetro de regularización de la máquina de vectores de soporte, el inverso del \(\lambda\) de la regresión ridge de la Unidad 2 y, al igual que \(k\) en k-NN, se elige por validación cruzada. El bloque de código ajusta máquinas lineales con \(C = 0.01\), 1 y 100 al problema de dos características; <code>svm</code> estandariza internamente las características por defecto.



In [ ]:
# El efecto de C en una SVM lineal de margen blando (dos características): ancho del margen, vectores de soporte, exactitud.
svm_region_bc <- function(fit, main, res = 120) {
  gx <- seq(xlim[1], xlim[2], length.out = res); gy <- seq(ylim[1], ylim[2], length.out = res)
  G <- setNames(expand.grid(gx, gy), feat2)
  image(gx, gy, matrix(as.integer(predict(fit, G) == "malignant"), res), col = c("#dfe7f1", "#fbe3d3"),
        main = main, xlab = "clump thickness", ylab = "cell size")
  dv <- attr(predict(fit, G, decision.values = TRUE), "decision.values")[, 1]
  contour(gx, gy, matrix(dv, res), levels = c(-1, 0, 1), add = TRUE, drawlabels = FALSE, lty = c(2, 1, 2))
  points(jitter(tr[[feat2[1]]]), jitter(tr[[feat2[2]]]), col = pcol(tr), pch = 19, cex = 0.4)
}
par(mfrow = c(1, 3), mar = c(4, 4, 3, 1))
for (C in c(0.01, 1, 100)) {
  fit <- svm(f2, data = tr, kernel = "linear", cost = C)
  svm_region_bc(fit, sprintf("C = %g: %d SVs, test %.3f", C, fit$tot.nSV, mean(predict(fit, te) == te$Class)))
}



<p class="caption"><b>Figura 14.</b> Máquinas lineales de margen blando con \(C = 0.01\), 1 y 100; línea continua, la frontera; líneas discontinuas, los márgenes. Con \(C = 0.01\) el margen es enorme y casi todos los casos son vectores de soporte; con \(C = 1\) y 100 el corredor es estrecho, solo los casos de la zona de superposición lo sostienen, y las dos soluciones son casi idénticas, porque una vez que \(C\) es lo bastante grande como para que las violaciones sean costosas, los mismos pocos casos superpuestos determinan la frontera. Las exactitudes de prueba difieren poco en este problema fácil, lo cual es una lección en sí misma: \(C\) importa sobre todo cuando las clases se superponen mucho o la muestra es pequeña.</p>

### 7.3 El truco del kernel

Una frontera lineal no puede separar clases dispuestas en anillos concéntricos o en medialunas entrelazadas. El remedio que hace general a la máquina de vectores de soporte es la observación, debida a Boser, Guyon y Vapnik en 1992,<sup class="c"><a href="#r24">[24]</a></sup> de que el problema de entrenamiento y la función de decisión dependen de los datos solo a través de los productos internos \(\mathbf{x}_i^\top \mathbf{x}_j\). Si los datos se transforman primero a un espacio de mayor dimensión mediante alguna \(\phi(\mathbf{x})\), una frontera lineal en ese espacio es una frontera curva en el espacio original, y los productos internos pasan a ser \(\phi(\mathbf{x}_i)^\top \phi(\mathbf{x}_j)\). Un **kernel** \(K(\mathbf{x}_i, \mathbf{x}_j)\) calcula directamente este producto interno, sin construir nunca \(\phi\); el teorema de Mercer de 1909 caracteriza las funciones para las cuales existe una \(\phi\) así.<sup class="c"><a href="#r27">[27]</a></sup> La función de decisión pasa a ser \(f(\mathbf{x}) = \sum_{i \in \text{SV}} \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}) + b\), una suma ponderada de similitudes entre el caso nuevo y los vectores de soporte. Los kernels habituales son el kernel **lineal**, el kernel **polinomial** \((\gamma\, \mathbf{x}_i^\top \mathbf{x}_j + r)^d\) y el kernel de **función de base radial (RBF)**

\[ K(\mathbf{x}_i, \mathbf{x}_j) = \exp\!\big(-\gamma\, \lVert \mathbf{x}_i - \mathbf{x}_j \rVert^2\big), \]

cuyo espacio de características implícito tiene dimensión infinita. Un cuarto kernel, el **sigmoide** \(\tanh(\gamma\, \mathbf{x}_i^\top \mathbf{x}_j + r)\), hace que la función de decisión se parezca a una red neuronal de dos capas; no es un kernel de Mercer válido para todos los parámetros y rara vez es la mejor opción. La monografía de Schölkopf y Smola es la referencia estándar.<sup class="c"><a href="#r28">[28]</a></sup> El bloque de código muestra los kernels en los dos problemas no lineales clásicos.



In [ ]:
# Kernels lineal, polinomial y RBF (y sigmoide) en dos problemas no linealmente separables (medialunas y círculos).
svm_panel <- function(fit, X, y, xlim, ylim, main, res = 120) {
  gx <- seq(xlim[1], xlim[2], length.out = res); gy <- seq(ylim[1], ylim[2], length.out = res); G <- as.matrix(expand.grid(gx, gy))
  dv <- attr(predict(fit, G, decision.values = TRUE), "decision.values")[, 1]
  image(gx, gy, matrix(as.integer(predict(fit, G) == levels(y)[2]), res), col = c("#fbe3d3", "#dfe7f1"),
        main = main, xlab = "", ylab = "", xaxt = "n", yaxt = "n")
  contour(gx, gy, matrix(dv, res), levels = 0, add = TRUE, drawlabels = FALSE, lwd = 1.2)
  points(X, col = ifelse(y == levels(y)[1], ORANGE, BLUE), pch = 19, cex = 0.4)
}
probs <- list(moons = make_moons(300, 0.25), circles = make_circles(300, 0.1, 0.4))
kerns <- list(c("linear", "linear"), c("poly, deg 3", "polynomial"), c("RBF, gamma 1", "radial"), c("sigmoid", "sigmoid"))
par(mfrow = c(2, 4), mar = c(1.5, 1.5, 3, 0.8))
for (pn in names(probs)) {
  X <- probs[[pn]]$X; y <- probs[[pn]]$y; set.seed(0); ti <- sample(nrow(X), 0.7 * nrow(X))
  xl <- range(X[, 1]) + c(-.4, .4); yl <- range(X[, 2]) + c(-.4, .4)
  for (kn in kerns) {
    fit <- svm(X[ti, ], y[ti], kernel = kn[2], cost = 1, gamma = if (kn[2] == "radial") 1 else if (kn[2] == "sigmoid") 0.5 else 1, degree = 3)
    svm_panel(fit, X[ti, ], y[ti], xl, yl, sprintf("%s, %s\ntest %.2f", pn, kn[1], mean(predict(fit, X[-ti, ]) == y[-ti])))
  }
}



<p class="caption"><b>Figura 15.</b> Dos problemas no lineales (filas) y cuatro kernels (columnas), con los puntos de entrenamiento y la frontera de decisión. El kernel lineal no puede hacer nada mejor que un corte recto; el polinomio cúbico separa bien las medialunas, pero describe mal los anillos; el kernel RBF separa ambos con alta exactitud; el kernel sigmoide produce una frontera con forma propia y no separa bien ninguno de los dos con estos parámetros, lo cual es representativo de su reputación. El kernel RBF es la opción por defecto de <code>svm</code> y de LIBSVM,<sup class="c"><a href="#r33">[33]</a></sup> y la primera elección habitual cuando no se sabe nada sobre la forma de la frontera.</p>

Por qué una transformación no lineal vuelve lineal un problema no lineal puede verse sin ningún kernel, con la imagen de "elevación" de las diapositivas de clase. Los anillos no son linealmente separables en el plano \((x_1, x_2)\). Añade una tercera característica \(z = x_1^2 + x_2^2\) —la distancia al centro al cuadrado— y el disco interior se queda abajo mientras el anillo exterior sube, de modo que un umbral sobre \(z\) los separa. El bloque de código realiza la elevación y ajusta una máquina lineal antes y después.



In [ ]:
# Elevar un problema no linealmente separable a un espacio donde es linealmente separable: añadir z = x1^2 + x2^2.
ci <- make_circles(300, 0.08, 0.4); Xc <- ci$X; yc <- ci$y; z <- rowSums(Xc^2)
lin2 <- svm(Xc, yc, kernel = "linear", cost = 1); lin3 <- svm(cbind(Xc, z), yc, kernel = "linear", cost = 1)
cat(sprintf("linear SVM training accuracy: in (x1,x2) = %.3f | in (x1,x2,z) = %.3f\n",
            mean(predict(lin2, Xc) == yc), mean(predict(lin3, cbind(Xc, z)) == yc)))
zcut <- mean(c(mean(z[yc == "inner"]), mean(z[yc == "outer"])))
par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
plot(Xc, col = ifelse(yc == "outer", ORANGE, BLUE), pch = 19, cex = 0.5, asp = 1,
     xlab = "x1", ylab = "x2", main = "Original plane: not linearly separable")
plot(Xc[, 1], z, col = ifelse(yc == "outer", ORANGE, BLUE), pch = 19, cex = 0.5,
     xlab = "x1", ylab = "z = x1^2 + x2^2", main = "Lifted feature z: a threshold separates the classes")
abline(h = zcut, col = GREEN, lwd = 2, lty = 2)
legend("topright", c("outer ring", "inner disc", "separating z"), col = c(ORANGE, BLUE, GREEN),
       pch = c(19, 19, NA), lty = c(NA, NA, 2), bty = "n", cex = 0.8)



<p class="caption"><b>Figura 16.</b> Izquierda: los anillos en sus coordenadas originales, donde una máquina lineal es apenas mejor que el azar (exactitud de entrenamiento impresa). Derecha: los mismos puntos con \(z = x_1^2 + x_2^2\) en el eje vertical; el disco interior se ubica en valores bajos de \(z\), el anillo exterior en valores altos de \(z\), y la línea verde (una frontera lineal en el espacio elevado) los separa; de vuelta en el plano original, esa línea es un círculo. Esta vista bidimensional reemplaza el gráfico tridimensional de la unidad en Python; el truco del kernel logra lo mismo para transformaciones demasiado grandes para escribirlas explícitamente, calculando solo productos internos.</p>

### 7.4 Gamma y las dos perillas de una máquina RBF

El kernel RBF tiene un parámetro de ancho, \(\gamma\). Un **\(\gamma\) pequeño** hace que cada protuberancia gaussiana sea ancha, de modo que puntos lejanos influyen en la decisión y la frontera es suave, casi lineal. Un **\(\gamma\) grande** hace que las protuberancias sean estrechas, de modo que solo importan los puntos cercanos y la frontera sigue a casos individuales. En conjunto, \(C\) y \(\gamma\) controlan la complejidad de una máquina RBF, y ambos se ajustan por validación cruzada sobre una grilla logarítmica; la guía práctica de LIBSVM recomienda exactamente esto, precedido de escalado.<sup class="c"><a href="#r33">[33]</a></sup> El bloque de código dibuja la frontera para cada combinación de tres valores de cada uno en las medialunas.



In [ ]:
# Una grilla de fronteras de decisión: C (filas) x gamma (columnas) para el kernel RBF en las medialunas.
mo <- make_moons(300, 0.25); Xm <- mo$X; ym <- mo$y; set.seed(0); ti <- sample(nrow(Xm), 0.7 * nrow(Xm))
xl <- range(Xm[, 1]) + c(-.4, .4); yl <- range(Xm[, 2]) + c(-.4, .4)
par(mfrow = c(3, 3), mar = c(1.2, 1.2, 2.6, 0.6))
for (C in c(0.1, 1, 100)) for (g in c(0.1, 1, 50)) {
  fit <- svm(Xm[ti, ], ym[ti], kernel = "radial", cost = C, gamma = g)
  svm_panel(fit, Xm[ti, ], ym[ti], xl, yl,
            sprintf("C=%g g=%g | test %.2f, SV %d", C, g, mean(predict(fit, Xm[-ti, ]) == ym[-ti]), fit$tot.nSV))
}



<p class="caption"><b>Figura 17.</b> Máquinas RBF en las medialunas para \(C \in \{0.1, 1, 100\}\) (filas) y \(\gamma \in \{0.1, 1, 50\}\) (columnas); cada título da la exactitud de prueba y el número de vectores de soporte. A lo largo de una fila, aumentar \(\gamma\) cambia la frontera de casi recta a un conjunto de islas alrededor de casos individuales. A lo largo de una columna, aumentar \(C\) estrecha el margen y elimina errores de clasificación en entrenamiento a costa de una frontera más retorcida. La máquina de arriba a la izquierda subajusta, la de abajo a la derecha memoriza (muchos vectores de soporte, menor exactitud de prueba: el equivalente en SVM de \(k = 1\) en la Figura 11), y la mejor exactitud de prueba se encuentra en el interior de la grilla. Una máquina que conserva la mayoría de sus casos de entrenamiento como vectores de soporte los ha memorizado: una señal de alerta útil.</p>

### 7.5 Probabilidades, regresión y la pérdida bisagra

Una máquina de vectores de soporte entrega una distancia con signo a la frontera, \(f(\mathbf{x})\), no una probabilidad. Platt propuso ajustar una función logística \(P(y = 1 \mid \mathbf{x}) = 1 / (1 + \exp(A f(\mathbf{x}) + B))\) a las salidas sobre datos reservados: el **escalado de Platt** (Platt scaling);<sup class="c"><a href="#r29">[29]</a></sup> Lin, Lin y Weng dieron el procedimiento de ajuste numéricamente robusto que implementa LIBSVM y, por lo tanto, <code>e1071::svm(probability = TRUE)</code>.<sup class="c"><a href="#r30">[30]</a></sup> En una gran comparación empírica, Niculescu-Mizil y Caruana encontraron que los puntajes de SVM calibrados de esta manera dan buenas probabilidades, mientras que las probabilidades a posteriori sin procesar de Bayes ingenuo están entre las peor calibradas.<sup class="c"><a href="#r31">[31]</a></sup> La idea del margen se traslada a la regresión invirtiéndola: la **regresión de vectores de soporte** (SVR) ajusta una función que se mantiene dentro de un tubo de semiancho \(\varepsilon\) alrededor de los valores objetivo, penalizando solo los puntos fuera del tubo, con el mismo \(C\) y los mismos kernels; el tutorial de Smola y Schölkopf es la referencia.<sup class="c"><a href="#r32">[32]</a></sup>

Reescribir el problema de margen blando sin restricciones muestra qué minimiza la máquina. Como \(\xi_i = \max(0, 1 - y_i f(\mathbf{x}_i))\) en el óptimo, el problema es una **pérdida bisagra** (hinge loss) más una penalización \(L_2\), \(\sum_i \max(0, 1 - y_i f(\mathbf{x}_i)) + \tfrac{1}{2C} \lVert \mathbf{w} \rVert^2\): la misma estructura de "pérdida más regularización" que la regresión logística de la Unidad 2, cuya pérdida es \(\log(1 + e^{-y_i f(\mathbf{x}_i)})\). El bloque de código dibuja las dos pérdidas junto con la pérdida cero-uno que aproximan, y luego demuestra el escalado de Platt y la SVR con datos reales.



In [ ]:
# Escalado de Platt en la SVM de cáncer de mama; SVR con BostonHousing; luego las tres pérdidas en función del margen.
svm_bc <- svm(Class ~ ., data = tr, kernel = "radial", cost = 1, probability = TRUE)
pr <- predict(svm_bc, te, decision.values = TRUE, probability = TRUE)
dv <- attr(pr, "decision.values")[, 1]; pb <- attr(pr, "probabilities")[, "benign"]
pick <- order(dv)[c(1, round(length(dv) / 4), round(length(dv) / 2), round(3 * length(dv) / 4), length(dv))]
cat("Platt-scaled P(benign) vs decision value f(x), five test cases from most to least benign:\n")
print(data.frame(f_x = round(dv[pick], 2), P_benign = round(pb[pick], 3),
                 truth = as.character(te$Class[pick])), row.names = FALSE)
set.seed(1); bf <- sample(rep(1:5, length.out = nrow(BostonHousing)))
svr_r2 <- sapply(c(1, 10, 100, 1000), function(C) {
  prv <- numeric(nrow(BostonHousing))
  for (f in 1:5) { tri <- bf != f
    prv[!tri] <- predict(svm(medv ~ ., BostonHousing[tri, ], kernel = "radial", cost = C, epsilon = 0.1), BostonHousing[!tri, ]) }
  round(1 - sum((BostonHousing$medv - prv)^2) / sum((BostonHousing$medv - mean(BostonHousing$medv))^2), 3) })
cat(sprintf("SVR (RBF, epsilon = 0.1) on BostonHousing, CV R2 by C: %s\n", paste(c(1, 10, 100, 1000), svr_r2, sep = ":", collapse = "  ")))
# La pérdida bisagra, la pérdida logística y la pérdida cero-uno, como funciones del margen.
m <- seq(-3, 3, length.out = 400)
plot(m, as.numeric(m < 0), type = "l", col = GREY, lwd = 1.6, ylim = c(0, 4),
     xlab = "margin  m = y * f(x)  (>1: outside the corridor)", ylab = "loss", main = "Three losses on the margin")
lines(m, pmax(0, 1 - m), col = ORANGE, lwd = 2)
lines(m, log(1 + exp(-m)) / log(2), col = BLUE, lwd = 2)
abline(v = c(0, 1), col = c("black", ORANGE), lty = c(1, 3), lwd = c(0.6, 1))
legend("topright", c("zero-one loss", "hinge loss (SVM)", "logistic loss"),
       col = c(GREY, ORANGE, BLUE), lwd = 2, bty = "n", cex = 0.8)



<p class="caption"><b>Figura 18.</b> Pérdida en función del margen \(m = y f(\mathbf{x})\). La pérdida bisagra es cero para los casos fuera del corredor (\(m \geq 1\)) —no ejercen ninguna fuerza sobre la solución, y por eso solo importan los vectores de soporte— y crece linealmente dentro de él. La pérdida logística nunca es exactamente cero: cada caso sigue tirando un poco de la frontera, y por eso la regresión logística no tiene vectores de soporte. Ambas son cotas superiores convexas de la pérdida cero-uno. La salida impresa del bloque de código muestra primero las probabilidades con escalado de Platt y los resultados de SVR que se comentan a continuación.</p>

Las probabilidades con escalado de Platt aumentan monótonamente con el valor de decisión y pasan por 0.5 cerca de \(f(\mathbf{x}) = 0\), convirtiendo una distancia en una probabilidad a la que se puede aplicar un umbral según los costos, como en la Sección 3.2. Los resultados de SVR muestran un regresor flexible con los datos de viviendas: su \(R^2\) aumenta con \(C\) hasta un valor alto, comparable o superior a los resultados lineales y de k-NN de la Sección 6.6. Las máquinas de vectores de soporte se usan en medicina desde fines de la década de 1990, con mayor influencia en la clasificación de tumores a partir de perfiles de expresión génica, donde el número de características supera con creces el número de pacientes y un clasificador de margen máximo con eliminación recursiva de características resultó ser a la vez exacto e interpretable.<sup class="c"><a href="#r34">[34]</a></sup>

<div class="box key"><h4>Idea clave</h4><p>Una máquina de vectores de soporte queda definida por tres elecciones —el kernel (forma de la frontera), \(C\) (tolerancia a las violaciones en entrenamiento) y, para el kernel RBF, \(\gamma\) (suavidad)—, y su solución depende solo de los vectores de soporte. El método no es probabilístico, escala mal más allá de algunas decenas de miles de casos (el entrenamiento es entre cuadrático y cúbico en \(n\)) y exige escalar las características; a cambio, está entre los clasificadores listos para usar más exactos en problemas pequeños, de alta dimensión y ruidosos, lo que describe a la mayoría de los conjuntos de datos biomédicos.</p></div>

<div class="exercise"><h4>Ejercicio 4</h4><p>En el bloque de código de kernels de la Sección 7.3, aumenta el ruido de las medialunas a 0.4 y vuelve a ajustar. ¿Qué kernel se degrada más? Luego fija <code>cost = 100</code> para la máquina RBF y compara la exactitud de entrenamiento y de prueba. Explica el resultado usando las variables de holgura de la Sección 7.2.</p></div>

## 8. Comparación de las tres familias

### 8.1 Una comparación honesta con los datos completos

Cada exactitud citada hasta ahora provino de una sola división o de una validación cruzada en la que los hiperparámetros se habían elegido mirando los mismos pliegues. El Suplemento S1 explica por qué lo segundo es optimista: elegir \(k\) o \(C\) para maximizar el puntaje de validación cruzada y luego informar ese puntaje usa dos veces los datos de validación. El procedimiento honesto es la **validación cruzada anidada**: una validación cruzada interna dentro de cada pliegue de entrenamiento elige los hiperparámetros, y el pliegue externo, nunca visto por la búsqueda interna, mide el desempeño. El bloque de código la ejecuta con 5 pliegues externos estratificados para las tres familias, con la regresión logística de la Unidad 2 como referencia, sobre las nueve características, e informa la media ± desviación estándar de la exactitud y del área bajo la curva ROC (AUC).



In [ ]:
# Validación cruzada anidada de 5 pliegues de las tres familias (y de la regresión logística) con las nueve características.
X9 <- bc[, 1:9]; yv <- bc$Class; set.seed(7); of <- sample(rep(1:5, length.out = nrow(bc)))
auc1 <- function(truth, score) as.numeric(pROC::auc(truth, score, levels = c("benign", "malignant"), direction = "<", quiet = TRUE))
mods <- c("Gaussian naive Bayes", "k-NN (k tuned)", "SVM linear (C tuned)", "SVM RBF (C,gamma tuned)", "logistic regression")
A <- Au <- matrix(NA, 5, 5, dimnames = list(NULL, mods))
for (f in 1:5) {
  tri <- of != f; Xtr <- X9[tri, ]; ytr <- yv[tri]; Xte <- X9[!tri, ]; yte <- yv[!tri]
  ct <- colMeans(Xtr); s <- apply(Xtr, 2, sd); Xtrs <- scale(Xtr, ct, s); Xtes <- scale(Xte, ct, s)
  nb <- naiveBayes(Xtr, ytr); A[f, 1] <- mean(predict(nb, Xte) == yte); Au[f, 1] <- auc1(yte, predict(nb, Xte, type = "raw")[, "malignant"])
  tk <- e1071::tune.knn(as.matrix(Xtrs), ytr, k = c(3, 5, 9, 15, 25))$best.parameters$k
  pk <- knn(Xtrs, Xtes, ytr, k = tk, prob = TRUE); pw <- attr(pk, "prob")
  A[f, 2] <- mean(pk == yte); Au[f, 2] <- auc1(yte, ifelse(pk == "malignant", pw, 1 - pw))
  sl <- e1071::tune.svm(Xtr, ytr, kernel = "linear", cost = c(0.1, 1, 10), probability = TRUE)$best.model
  A[f, 3] <- mean(predict(sl, Xte) == yte); Au[f, 3] <- auc1(yte, attr(predict(sl, Xte, probability = TRUE), "probabilities")[, "malignant"])
  sr <- e1071::tune.svm(Xtr, ytr, kernel = "radial", cost = c(1, 10, 100), gamma = c(0.03, 0.1, 0.3), probability = TRUE)$best.model
  A[f, 4] <- mean(predict(sr, Xte) == yte); Au[f, 4] <- auc1(yte, attr(predict(sr, Xte, probability = TRUE), "probabilities")[, "malignant"])
  lr <- suppressWarnings(glm(Class ~ ., data = bc[tri, ], family = binomial)); pl <- predict(lr, bc[!tri, ], type = "response")
  A[f, 5] <- mean(ifelse(pl > 0.5, "malignant", "benign") == yte); Au[f, 5] <- auc1(yte, pl)
}
cmp <- data.frame(accuracy = sprintf("%.3f +/- %.3f", colMeans(A), apply(A, 2, sd)),
                  AUC = sprintf("%.3f +/- %.3f", colMeans(Au), apply(Au, 2, sd)), row.names = mods)
print(cmp)



Los cinco métodos clasifican bien los datos de Wisconsin —todos los AUC están cerca de 0.99—, pero las exactitudes no son todas iguales. Los cinco métodos alcanzan exactitudes en torno a 0.96–0.97 que difieren en menos que la desviación estándar entre pliegues, de modo que la tabla no da fundamentos para preferir a uno de ellos. Bayes ingenuo, sin hiperparámetros que ajustar, es el más rápido y tiene el AUC más bajo de los cinco (aunque por un margen menor que su desviación estándar): en coherencia con la Sección 4.5, ordena los casos de forma algo menos nítida debido a su supuesto de independencia sobre características correlacionadas, y aun así clasifica con tanta exactitud como los demás en este problema bien separado. La lección general es típica de los problemas pequeños, bien curados y de baja dimensión: los métodos están cerca, y las desviaciones estándar entre pliegues —del orden de 0.01–0.02— son la resolución por debajo de la cual no debe hacerse ninguna afirmación a partir de una muestra de 683 casos. Donde los métodos difieren de forma decisiva es en lo que ofrecen además de la exactitud, como se tabula a continuación.

### 8.2 Cuándo usar cada uno

| | Bayes ingenuo | Red bayesiana | k-NN | SVM |
|---|---|---|---|---|
| **Familia** | generativo, probabilístico | generativo, probabilístico | basado en memoria | discriminativo, geométrico |
| **Supuesto principal** | características condicionalmente independientes dada la clase | la estructura de dependencia del grafo | los casos cercanos comparten etiquetas; existe una distancia con sentido | clases separables con un margen en el espacio de características del kernel |
| **Forma de la frontera** | cuadrática (gaussiano), lineal (varianzas iguales) | depende del modelo | arbitraria, local | lineal o tan flexible como el kernel |
| **Requiere escalar las características** | no | no | **sí** | **sí** |
| **Hiperparámetros** | ninguno (suavizado) | estructura, distribuciones a priori | \(k\), métrica, ponderación | kernel, \(C\), \(\gamma\) |
| **Entrega probabilidades** | sí, mal calibradas | sí, y distribuciones a posteriori completas | por fracción de votos, groseras | no; requiere escalado de Platt |
| **Costo de entrenamiento** | una pasada, \(O(np)\) | conteo (estructura conocida); búsqueda (desconocida) | ninguno | \(O(n^2)\)–\(O(n^3)\) |
| **Costo de predicción** | \(O(p)\) | inferencia, exponencial en el peor caso | \(O(np)\) por consulta | \(O(\#\text{SV} \cdot p)\) |
| **Interpretabilidad** | verosimilitudes por característica | alta: el grafo es la explicación | mediante ejemplos | lineal: coeficientes; con kernel: baja |
| **Tamaño de muestra** | funciona con muy pocos casos | pequeño si se conoce la estructura | necesita un muestreo denso | bueno con \(n\) pequeño y \(p\) grande |
| **Maneja valores faltantes** | de forma natural (omite la característica) | de forma natural (marginaliza) | requiere imputación | requiere imputación |
| **Uso biomédico típico** | texto, modelos de referencia, diagnóstico con muestras pequeñas | diagnóstico con conocimiento experto | imputación, búsqueda por similitud | expresión génica e imágenes, \(p\) alto |

De aquí se desprenden tres reglas prácticas. Usa Bayes ingenuo como primer modelo de referencia en cualquier problema nuevo y como candidato serio cuando los casos escasean o las características son conteos. Usa una red bayesiana cuando se conoce la estructura del problema y hay que razonar sobre evidencia observada parcialmente: cuando la pregunta no es solo "¿qué clase?", sino "¿cuál es la probabilidad de este hallazgo dados esos otros?". Usa k-NN cuando la similitud es la noción natural (recuperación, imputación) y los datos son densos en una representación de baja dimensión; usa una máquina de vectores de soporte cuando los datos son de alta dimensión, la muestra es pequeña y la exactitud importa más que las probabilidades o las explicaciones. En todos los casos, escala las características cuando el método se basa en distancias, ajusta el escalador dentro del remuestreo, ajusta los hiperparámetros mediante validación cruzada anidada e informa la dispersión entre pliegues.

<div class="box bridge"><h4>Puente con las Unidades 4 y 6</h4><p>La Unidad 4 añade una cuarta familia, los árboles de decisión y los bosques aleatorios, que no requieren escalado, manejan de forma nativa tipos de datos mixtos y producen reglas legibles, y que en datos biomédicos tabulares suelen ser el modelo de referencia más fuerte. La Unidad 6 muestra que una red neuronal con una capa oculta es también, en cierto modo, una máquina de kernel —aprende su propia transformación de características \(\phi\) en lugar de fijarla de antemano— y que las pérdidas bisagra y logística de la Figura 18 reaparecen como su capa de salida.</p></div>

## 9. Resumen

Ahora deberías ser capaz de explicar, sin apuntes:

- El teorema de Bayes como la relación entre la dirección directa, en la que se miden las características de una prueba, y la dirección inversa, en la que razona un clínico, y por qué un resultado positivo de una prueba exacta para una enfermedad poco frecuente (prevalencia del 1 %, sensibilidad y especificidad del 95 %) deja una probabilidad a posteriori de solo alrededor del 16 %.
- El clasificador de Bayes, el error de Bayes como el error irreducible de un problema, y cómo los costos desiguales reemplazan el umbral de probabilidad a posteriori de 1/2 por \(\lambda_{\mathrm{FP}} / (\lambda_{\mathrm{FP}} + \lambda_{\mathrm{FN}})\).
- El supuesto de independencia condicional de Bayes ingenuo; los tres pasos (tabla de frecuencias, tabla de verosimilitudes, probabilidad a posteriori) del ejemplo categórico a mano; los parámetros y la log-probabilidad a posteriori de la variante gaussiana; el suavizado de Laplace y el problema de la frecuencia cero; y la convención de varianza n frente a (n − 1) que distingue a <code>GaussianNB</code> de <code>e1071</code>.
- Por qué Bayes ingenuo clasifica bien aunque su supuesto sea falso (el orden se conserva mientras las probabilidades se distorsionan), y por qué no se debe confiar en sus probabilidades a posteriori sin calibrarlas.
- Una red bayesiana como un DAG más tablas de probabilidad condicional, la factorización que codifica, la inferencia por enumeración, la explicación alternativa (explaining away) y las tres estructuras de d-separación (cadena, bifurcación, colisionador).
- Bayes ingenuo como la red bayesiana con un solo padre; el aprendizaje de parámetros como conteo; la existencia y el costo del aprendizaje de la estructura.
- La regla k-NN como aprendiz perezoso, sus métricas de distancia, por qué el escalado de las características es obligatorio, cómo \(k\) controla el compromiso sesgo–varianza, qué le hace la maldición de la dimensionalidad a "más cercano", y por qué el paso costoso es la predicción y no el entrenamiento.
- El hiperplano de margen máximo y los vectores de soporte; el margen blando y el significado de \(C\); el truco del kernel, los kernels habituales y el significado de \(\gamma\); el escalado de Platt; la pérdida bisagra y su relación con la pérdida logística.
- Cómo comparar familias de clasificadores de forma honesta (validación cruzada anidada, media ± desviación estándar entre pliegues) y cómo elegir entre ellas según sus supuestos, sus necesidades de escalado, sus costos y su interpretabilidad.

## 10. Autoevaluación

<details><summary>1. Una prueba tiene sensibilidad 0.90 y especificidad 0.90. Calcula a mano el VPP cuando la prevalencia es del 2 % y luego cuando es del 50 %.</summary>Con 2 %: \(P(D \mid T) = (0.9 \times 0.02) / (0.9 \times 0.02 + 0.1 \times 0.98) = 0.018 / 0.116 \approx 0.155\). Con 50 %: \(0.45 / (0.45 + 0.05) = 0.90\). Solo cuando la probabilidad a priori es del 50 % el VPP es igual a la sensibilidad.</details>

<details><summary>2. En la Figura 4, el umbral de mínimo riesgo desplazó la frontera hacia valores menores de grosor del grumo. Si, en cambio, un falso positivo fuera cinco veces más costoso que un falso negativo, ¿hacia dónde se desplazaría y cuál sería el umbral de probabilidad a posteriori?</summary>El umbral pasa a ser \(\lambda_{\mathrm{FP}} / (\lambda_{\mathrm{FP}} + \lambda_{\mathrm{FN}}) = 5/6\): un caso se declara maligno solo cuando la probabilidad a posteriori supera 0.83, de modo que el umbral de grosor del grumo se desplaza hacia valores mayores y se declaran malignos menos casos.</details>

<details><summary>3. En el ejemplo del clima, calcula a mano \(P(\text{Yes} \mid \text{Overcast}, \text{Hot}, \text{High}, \text{Windy} = \text{False})\) sin suavizado y explica qué ocurre con la clase No.</summary>Yes: \(9/14 \times 4/9 \times 2/9 \times 3/9 \times 6/9 \approx 0.0141\). No: \(5/14 \times 0/5 \times \dots = 0\). La probabilidad a posteriori normalizada de Yes es exactamente 1, porque la única verosimilitud nula \(P(\text{Overcast} \mid \text{No}) = 0\) anula todo el producto para No: el problema de la frecuencia cero que evita el suavizado de Laplace.</details>

<details><summary>4. Bayes ingenuo gaussiano con probabilidades a priori 0.5, media benigna (12, 2) y varianzas pequeñas, media maligna (19, 9) y varianzas grandes. Sin hacer cálculos, di qué clase recibe el punto (12, 2) y qué clase recibe un punto muy alejado de ambas medias, y por qué.</summary>(12, 2) está en la media benigna y lejos de la maligna: benigno. Un punto muy alejado de ambas medias se asigna a la clase con <em>mayor varianza</em> (maligna), porque la penalización cuadrática \((x - \mu)^2 / 2\sigma^2\) crece más lentamente para la clase ancha; una gaussiana estrecha hace que los puntos lejanos sean más sorprendentes que una ancha. (Esta es también la razón por la que la respuesta con los seis casos queda sobre la frontera y cambia entre las convenciones de varianza n y n − 1).</details>

<details><summary>5. Bayes ingenuo alcanzó una exactitud de alrededor de 0.96 con las nueve características, aunque el tamaño celular, la forma celular y el tamaño de las células epiteliales están fuertemente correlacionados. ¿Por qué la redundancia no perjudicó la exactitud y qué perjudicó?</summary>Con pérdida cero-uno solo importa el orden de las probabilidades a posteriori; contar varias veces la misma evidencia infla las log-verosimilitudes de ambas clases de una manera que rara vez cambia cuál es mayor (Domingos y Pazzani). Lo que se perjudica es la calibración: las probabilidades a posteriori se empujan hacia 0 y 1, de modo que una gran fracción de los casos de prueba recibe una probabilidad a posteriori superior a 0.99.</details>

<details><summary>6. En la red de la clínica torácica, ¿es Smoking independiente de X-ray? ¿Es independiente de X-ray dado Cancer? ¿Es Cancer independiente de Bronchitis dado Dyspnoea? Justifica cada respuesta con el grafo.</summary>No: la cadena S → C → X está abierta, de modo que S y X son dependientes. Sí: condicionar en C bloquea la cadena. No: C → D ← B es un colisionador; condicionar en D abre el camino y hace dependientes a C y B (explicación alternativa), y además la bifurcación C ← S → B también está abierta, salvo que se observe S.</details>

<details><summary>7. Cuenta los parámetros libres de la red Asia completa (8 nodos binarios: Asia, Tuberculosis, Smoking, Cancer, Bronchitis, Either, X-ray, Dyspnoea) con las flechas A → T, S → C, S → B, T → E, C → E, E → X, E → D, B → D.</summary>Asia 1, Tuberculosis 2, Smoking 1, Cancer 2, Bronchitis 2, Either 4 (dos padres; determinista en el original), X-ray 2, Dyspnoea 4: 18 parámetros, frente a \(2^8 - 1 = 255\) para la conjunta sin restricciones.</details>

<details><summary>8. En la función auxiliar <code>knn_cv</code> de la Sección 6.3, el escalador se ajusta dentro de cada pliegue (<code>scale(Xtr, ct, s)</code> con estadísticos del pliegue de entrenamiento). ¿Por qué es necesario esto, en lugar de escalar toda la matriz una sola vez antes de la validación cruzada?</summary>Ajustar el escalador solo con el pliegue de entrenamiento mantiene la media y la varianza del pliegue de prueba fuera del modelo; escalar toda la matriz de antemano usaría los estadísticos de los casos de prueba: una forma (leve) de fuga de datos que vuelve optimista la estimación. El Suplemento S1 trata la fuga de datos en detalle.</details>

<details><summary>9. La Figura 10 muestra una exactitud de entrenamiento de exactamente 1.000 con \(k = 1\). ¿Se mantendría esto con ponderación por distancia para cualquier \(k\)? ¿Se mantendría con \(k = 1\) si los datos contuvieran dos casos idénticos con etiquetas distintas?</summary>Con pesos por distancia, el propio caso de entrenamiento está a distancia 0 y recibe un peso (en la práctica) infinito, de modo que la exactitud de entrenamiento es 1.000 para todo \(k\). Con dos casos idénticos de etiquetas distintas, \(k = 1\) no puede clasificar bien a ambos, de modo que la exactitud de entrenamiento cae por debajo de 1.</details>

<details><summary>10. En la Figura 13, el hiperplano de margen máximo estaba sostenido por muy pocos casos. Si uno de los casos que no son de soporte se moviera un poco, ¿cambiaría el hiperplano? ¿Y si se moviera un vector de soporte?</summary>Mover un caso que no es de soporte a cualquier lugar de su propio lado del margen deja la solución sin cambios: tiene peso \(\alpha_i\) nulo. Mover un vector de soporte cambia el hiperplano, porque la solución está determinada solo por los vectores de soporte.</details>

<details><summary>11. Explica, en términos de las variables de holgura y de la pérdida bisagra, por qué una SVM con \(C = 100\) sobre datos ruidosos tiene más vectores de soporte que una con \(C\) pequeño, como en la Figura 17.</summary>Con un \(C\) grande, el optimizador no está dispuesto a pagar por la holgura, de modo que estrecha el margen y retuerce la frontera (un \(\gamma\) grande se lo permite) para reducir las violaciones; muchos casos terminan sobre el margen estrecho o dentro de él, donde la pérdida bisagra está activa, y todos ellos se convierten en vectores de soporte. Con un \(C\) pequeño se acepta un margen ancho, las violaciones son baratas y menos casos determinan la frontera.</details>

<details><summary>12. La Sección 8.1 informa exactitudes de las máquinas de vectores de soporte, k-NN y la regresión logística que difieren en menos que la desviación estándar entre pliegues. Un colega propone elegir la de mayor media y presentarla como "el mejor modelo para el diagnóstico de cáncer de mama". ¿Qué está mal y qué informarías tú?</summary>Una diferencia menor que la variabilidad muestral no es evidencia de una diferencia; el orden probablemente cambiaría con otra división, y seleccionar el máximo de varias estimaciones ruidosas tiene en sí mismo un sesgo optimista. Informa todos los métodos con media ± desviación estándar, señala que son estadísticamente indistinguibles con 683 casos y elige según los criterios secundarios de la Sección 8.2 (calibración, interpretabilidad, costo), o aplica una prueba pareada entre pliegues si hay que defender una decisión.</details>

## Glosario

<dl class="gloss">
<dt>Clasificador de Bayes / error de Bayes (Bayes classifier / Bayes error)</dt><dd>La regla que asigna cada caso a la clase de mayor probabilidad a posteriori, óptima cuando se conocen las distribuciones verdaderas / su tasa de error, el error irreducible del problema.</dd>
<dt>Red bayesiana (Bayesian network)</dt><dd>Un grafo dirigido acíclico sobre variables aleatorias junto con una tabla de probabilidad condicional para cada nodo dados sus padres; codifica la factorización de la distribución conjunta en condicionales locales.</dd>
<dt>Calibración (calibration)</dt><dd>Concordancia entre las probabilidades predichas por un clasificador y las frecuencias observadas: entre los casos a los que se asigna probabilidad 0.8, alrededor del 80 % debería pertenecer a la clase.</dd>
<dt>Densidad condicional de clase (class-conditional density)</dt><dd>\(p(\mathbf{x} \mid y = c)\), la distribución de las características entre los casos de la clase \(c\); el objeto que estima un clasificador generativo.</dd>
<dt>Independencia condicional (conditional independence)</dt><dd>\(A\) y \(B\) son condicionalmente independientes dado \(C\) cuando \(P(A, B \mid C) = P(A \mid C)\,P(B \mid C)\); el supuesto que hace manejables a Bayes ingenuo y a las redes bayesianas.</dd>
<dt>Maldición de la dimensionalidad (curse of dimensionality)</dt><dd>Los fenómenos por los cuales la intuición geométrica falla a medida que crece el número de características: el volumen se concentra en capas externas y las distancias entre puntos se vuelven casi iguales.</dd>
<dt>d-separación (d-separation)</dt><dd>Un criterio gráfico para leer la independencia condicional en una red bayesiana: dos nodos son independientes dado un conjunto de nodos observados cuando todos los caminos entre ellos están bloqueados (las cadenas y bifurcaciones se bloquean al observar el nodo intermedio; los colisionadores están bloqueados salvo que se observe el colisionador o un descendiente suyo).</dd>
<dt>Discriminativo / generativo (discriminative / generative)</dt><dd>Un clasificador que modela \(P(y \mid \mathbf{x})\) directamente (regresión logística, SVM) / uno que modela \(p(\mathbf{x} \mid y)\) y \(P(y)\) y aplica el teorema de Bayes (Bayes ingenuo, redes bayesianas).</dd>
<dt>Explicación alternativa (explaining away)</dt><dd>La disminución de la probabilidad a posteriori de una causa cuando se descubre que está presente otra causa del mismo efecto observado; una consecuencia de condicionar en un colisionador.</dd>
<dt>Pérdida bisagra (hinge loss)</dt><dd>\(\max(0, 1 - y f(\mathbf{x}))\), la pérdida que minimiza la SVM de margen blando; cero para los casos fuera del margen, lineal dentro de él.</dd>
<dt>Kernel / truco del kernel (kernel / kernel trick)</dt><dd>Una función \(K(\mathbf{x}_i, \mathbf{x}_j)\) igual al producto interno de los dos puntos en algún espacio de características / la sustitución del producto interno por una función de ese tipo en un algoritmo, lo que lo vuelve no lineal sin construir el espacio de características.</dd>
<dt>k vecinos más cercanos (k-nearest neighbours, k-NN)</dt><dd>La regla que clasifica un caso por la etiqueta mayoritaria (o predice la media del valor objetivo) entre los \(k\) casos de entrenamiento más cercanos según una distancia elegida; un aprendiz perezoso.</dd>
<dt>Suavizado de Laplace (aditivo) (Laplace (additive) smoothing)</dt><dd>Sumar un pseudoconteo \(\alpha\) a cada celda de una tabla de frecuencias antes de estimar las probabilidades, para que los valores de características no observados no reciban probabilidad cero.</dd>
<dt>Aprendiz perezoso (lazy learner)</dt><dd>Un método que no construye ningún modelo durante el entrenamiento y posterga el cálculo hasta la predicción; k-NN es el arquetipo.</dd>
<dt>Margen (margin)</dt><dd>La distancia perpendicular desde un hiperplano separador hasta los casos de entrenamiento más cercanos; la SVM la maximiza.</dd>
<dt>Bayes ingenuo (naive Bayes)</dt><dd>El clasificador generativo que supone que las características son condicionalmente independientes dada la clase; las variantes gaussiana, multinomial, de Bernoulli y categórica difieren en el modelo unidimensional de cada característica.</dd>
<dt>Escalado de Platt (Platt scaling)</dt><dd>Ajustar una función logística a los puntajes de un clasificador sobre datos reservados para convertirlos en probabilidades calibradas; la forma estándar de obtener probabilidades a partir de una SVM.</dd>
<dt>Probabilidad a posteriori / a priori / verosimilitud / evidencia (posterior / prior / likelihood / evidence)</dt><dd>Los cuatro términos del teorema de Bayes: \(P(\text{class} \mid \text{data})\) / \(P(\text{class})\) antes de los datos / \(P(\text{data} \mid \text{class})\) / \(P(\text{data})\), la constante de normalización.</dd>
<dt>Kernel de función de base radial (radial basis function kernel, RBF)</dt><dd>\(K = \exp(-\gamma \lVert \mathbf{x}_i - \mathbf{x}_j \rVert^2)\); el kernel por defecto de la SVM, cuyo ancho se fija mediante \(\gamma\).</dd>
<dt>Margen blando / C (soft margin / C)</dt><dd>La formulación de la SVM que permite violaciones del margen medidas por variables de holgura / el parámetro que les pone precio, intercambiando ancho del margen por errores de entrenamiento.</dd>
<dt>Vector de soporte (support vector)</dt><dd>Un caso de entrenamiento que se encuentra sobre el margen o dentro de él (o que está mal clasificado); los únicos casos que determinan la solución de la SVM.</dd>
<dt>Curva de validación (validation curve)</dt><dd>Un gráfico del desempeño de entrenamiento y en validación cruzada frente a un hiperparámetro, usado para elegir su valor y para ver el subajuste y el sobreajuste.</dd>
</dl>

## Referencias y lecturas recomendadas

<ol class="refs">
<li id="r1">Bayes T, Price R. An essay towards solving a problem in the doctrine of chances. <em>Philos Trans R Soc Lond</em>. 1763;53:370–418. <a href="https://doi.org/10.1098/rstl.1763.0053">doi:10.1098/rstl.1763.0053</a></li>
<li id="r2">Casscells W, Schoenberger A, Graboys TB. Interpretation by physicians of clinical laboratory results. <em>N Engl J Med</em>. 1978;299(18):999–1001. <a href="https://doi.org/10.1056/NEJM197811022991808">doi:10.1056/NEJM197811022991808</a></li>
<li id="r3">Gigerenzer G, Hoffrage U. How to improve Bayesian reasoning without instruction: frequency formats. <em>Psychol Rev</em>. 1995;102(4):684–704. <a href="https://doi.org/10.1037/0033-295X.102.4.684">doi:10.1037/0033-295X.102.4.684</a></li>
<li id="r4">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r5">Domingos P, Pazzani M. On the optimality of the simple Bayesian classifier under zero-one loss. <em>Mach Learn</em>. 1997;29:103–130. <a href="https://doi.org/10.1023/A:1007413511361">doi:10.1023/A:1007413511361</a></li>
<li id="r6">Hand DJ, Yu K. Idiot's Bayes — not so stupid after all? <em>Int Stat Rev</em>. 2001;69(3):385–398. <a href="https://doi.org/10.1111/j.1751-5823.2001.tb00465.x">doi:10.1111/j.1751-5823.2001.tb00465.x</a></li>
<li id="r7">Ng AY, Jordan MI. On discriminative vs. generative classifiers: a comparison of logistic regression and naive Bayes. In: <em>Advances in Neural Information Processing Systems 14 (NIPS 2001)</em>. MIT Press; 2002:841–848. <a href="https://proceedings.neurips.cc/paper/2001/hash/7b7a53e239400a13bd6be6c91c4f6c4e-Abstract.html">proceedings.neurips.cc</a> (artículo de congreso sin DOI; página oficial de las actas)</li>
<li id="r8">Kononenko I. Machine learning for medical diagnosis: history, state of the art and perspective. <em>Artif Intell Med</em>. 2001;23(1):89–109. <a href="https://doi.org/10.1016/S0933-3657(01)00077-X">doi:10.1016/S0933-3657(01)00077-X</a></li>
<li id="r9">Pearl J. Fusion, propagation, and structuring in belief networks. <em>Artif Intell</em>. 1986;29(3):241–288. <a href="https://doi.org/10.1016/0004-3702(86)90072-X">doi:10.1016/0004-3702(86)90072-X</a></li>
<li id="r10">Pearl J. <em>Probabilistic Reasoning in Intelligent Systems: Networks of Plausible Inference</em>. Morgan Kaufmann; 1988. <a href="https://doi.org/10.1016/C2009-0-27609-4">doi:10.1016/C2009-0-27609-4</a></li>
<li id="r11">Lauritzen SL, Spiegelhalter DJ. Local computations with probabilities on graphical structures and their application to expert systems. <em>J R Stat Soc Ser B</em>. 1988;50(2):157–194. <a href="https://doi.org/10.1111/j.2517-6161.1988.tb01721.x">doi:10.1111/j.2517-6161.1988.tb01721.x</a></li>
<li id="r12">Quinlan JR. Induction of decision trees. <em>Mach Learn</em>. 1986;1:81–106. (Fuente del conjunto de datos del clima de catorce días). <a href="https://doi.org/10.1007/BF00116251">doi:10.1007/BF00116251</a></li>
<li id="r13">Heckerman D. A tutorial on learning with Bayesian networks. In: Jordan MI, ed. <em>Learning in Graphical Models</em>. Springer; 1998:301–354. <a href="https://doi.org/10.1007/978-94-011-5014-9_11">doi:10.1007/978-94-011-5014-9_11</a></li>
<li id="r14">Friedman N, Geiger D, Goldszmidt M. Bayesian network classifiers. <em>Mach Learn</em>. 1997;29:131–163. <a href="https://doi.org/10.1023/A:1007465528199">doi:10.1023/A:1007465528199</a></li>
<li id="r15">Lucas PJF, van der Gaag LC, Abu-Hanna A. Bayesian networks in biomedicine and health-care. <em>Artif Intell Med</em>. 2004;30(3):201–214. <a href="https://doi.org/10.1016/j.artmed.2003.11.001">doi:10.1016/j.artmed.2003.11.001</a></li>
<li id="r16">Fix E, Hodges JL. Discriminatory analysis. Nonparametric discrimination: consistency properties. USAF School of Aviation Medicine, Report 4; 1951. Reimpreso en <em>Int Stat Rev</em>. 1989;57(3):238–247. <a href="https://doi.org/10.2307/1403797">doi:10.2307/1403797</a></li>
<li id="r17">Cover T, Hart P. Nearest neighbor pattern classification. <em>IEEE Trans Inf Theory</em>. 1967;13(1):21–27. <a href="https://doi.org/10.1109/TIT.1967.1053964">doi:10.1109/TIT.1967.1053964</a></li>
<li id="r18">Dudani SA. The distance-weighted k-nearest-neighbor rule. <em>IEEE Trans Syst Man Cybern</em>. 1976;SMC-6(4):325–327. <a href="https://doi.org/10.1109/TSMC.1976.5408784">doi:10.1109/TSMC.1976.5408784</a></li>
<li id="r19">Altman NS. An introduction to kernel and nearest-neighbor nonparametric regression. <em>Am Stat</em>. 1992;46(3):175–185. <a href="https://doi.org/10.1080/00031305.1992.10475879">doi:10.1080/00031305.1992.10475879</a></li>
<li id="r20">Bellman RE. <em>Adaptive Control Processes: A Guided Tour</em>. Princeton University Press; 1961. <a href="https://doi.org/10.1515/9781400874668">doi:10.1515/9781400874668</a></li>
<li id="r21">Beyer K, Goldstein J, Ramakrishnan R, Shaft U. When is "nearest neighbor" meaningful? In: <em>Database Theory — ICDT'99</em>. Lecture Notes in Computer Science, vol 1540. Springer; 1999:217–235. <a href="https://doi.org/10.1007/3-540-49257-7_15">doi:10.1007/3-540-49257-7_15</a></li>
<li id="r22">Bentley JL. Multidimensional binary search trees used for associative searching. <em>Commun ACM</em>. 1975;18(9):509–517. <a href="https://doi.org/10.1145/361002.361007">doi:10.1145/361002.361007</a></li>
<li id="r23">Troyanskaya O, Cantor M, Sherlock G, et al. Missing value estimation methods for DNA microarrays. <em>Bioinformatics</em>. 2001;17(6):520–525. <a href="https://doi.org/10.1093/bioinformatics/17.6.520">doi:10.1093/bioinformatics/17.6.520</a></li>
<li id="r24">Boser BE, Guyon IM, Vapnik VN. A training algorithm for optimal margin classifiers. In: <em>Proceedings of the Fifth Annual Workshop on Computational Learning Theory (COLT '92)</em>. ACM; 1992:144–152. <a href="https://doi.org/10.1145/130385.130401">doi:10.1145/130385.130401</a></li>
<li id="r25">Cortes C, Vapnik V. Support-vector networks. <em>Mach Learn</em>. 1995;20:273–297. <a href="https://doi.org/10.1007/BF00994018">doi:10.1007/BF00994018</a></li>
<li id="r26">Vapnik VN. <em>The Nature of Statistical Learning Theory</em>. Springer; 1995. <a href="https://doi.org/10.1007/978-1-4757-2440-0">doi:10.1007/978-1-4757-2440-0</a></li>
<li id="r27">Mercer J. Functions of positive and negative type, and their connection with the theory of integral equations. <em>Philos Trans R Soc Lond A</em>. 1909;209:415–446. <a href="https://doi.org/10.1098/rsta.1909.0016">doi:10.1098/rsta.1909.0016</a></li>
<li id="r28">Schölkopf B, Smola AJ. <em>Learning with Kernels: Support Vector Machines, Regularization, Optimization, and Beyond</em>. MIT Press; 2002. <a href="https://doi.org/10.7551/mitpress/4175.001.0001">doi:10.7551/mitpress/4175.001.0001</a></li>
<li id="r29">Platt JC. Probabilistic outputs for support vector machines and comparisons to regularized likelihood methods. In: Smola AJ, Bartlett P, Schölkopf B, Schuurmans D, eds. <em>Advances in Large Margin Classifiers</em>. MIT Press; 2000:61–74. <a href="https://doi.org/10.7551/mitpress/1113.001.0001">doi:10.7551/mitpress/1113.001.0001</a> (DOI del volumen editado; el capítulo no tiene un DOI propio que hayamos podido verificar)</li>
<li id="r30">Lin H-T, Lin C-J, Weng RC. A note on Platt's probabilistic outputs for support vector machines. <em>Mach Learn</em>. 2007;68:267–276. <a href="https://doi.org/10.1007/s10994-007-5018-6">doi:10.1007/s10994-007-5018-6</a></li>
<li id="r31">Niculescu-Mizil A, Caruana R. Predicting good probabilities with supervised learning. In: <em>Proceedings of the 22nd International Conference on Machine Learning (ICML '05)</em>. ACM; 2005:625–632. <a href="https://doi.org/10.1145/1102351.1102430">doi:10.1145/1102351.1102430</a></li>
<li id="r32">Smola AJ, Schölkopf B. A tutorial on support vector regression. <em>Stat Comput</em>. 2004;14:199–222. <a href="https://doi.org/10.1023/B:STCO.0000035301.49549.88">doi:10.1023/B:STCO.0000035301.49549.88</a></li>
<li id="r33">Chang C-C, Lin C-J. LIBSVM: a library for support vector machines. <em>ACM Trans Intell Syst Technol</em>. 2011;2(3):27. <a href="https://doi.org/10.1145/1961189.1961199">doi:10.1145/1961189.1961199</a> (la "practical guide to SVM classification" de Hsu, Chang y Lin se distribuye con la biblioteca; <code>e1071::svm</code> es su interfaz en R)</li>
<li id="r34">Guyon I, Weston J, Barnhill S, Vapnik V. Gene selection for cancer classification using support vector machines. <em>Mach Learn</em>. 2002;46:389–422. <a href="https://doi.org/10.1023/A:1012487302797">doi:10.1023/A:1012487302797</a></li>
<li id="r35">Efron B, Hastie T, Johnstone I, Tibshirani R. Least angle regression. <em>Ann Stat</em>. 2004;32(2):407–499. (Fuente del conjunto de datos de diabetes que usa el curso en Python; esta edición en R usa <code>BostonHousing</code> para la regresión). <a href="https://doi.org/10.1214/009053604000000067">doi:10.1214/009053604000000067</a></li>
<li id="r36">R Core Team. R: A Language and Environment for Statistical Computing. R Foundation for Statistical Computing, Vienna. <a href="https://www.R-project.org/">https://www.R-project.org/</a> (software; sin DOI)</li>
<li id="r37">Meyer D, Dimitriadou E, Hornik K, Weingessel A, Leisch F. e1071: Misc Functions of the Department of Statistics, Probability Theory Group, TU Wien. Paquete de R (naiveBayes, svm, tune). <a href="https://doi.org/10.32614/CRAN.package.e1071">doi:10.32614/CRAN.package.e1071</a></li>
<li id="r38">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002. (Los paquetes <code>class</code> y <code>MASS</code>; <code>class::knn</code>). <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
<li id="r39">James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — El complemento en R de todo el curso; PDF gratuito en statlearning.com. <a href="https://doi.org/10.1007/978-1-0716-1418-1">doi:10.1007/978-1-0716-1418-1</a></li>
<li id="r40">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. Paquete de R (los datos Wisconsin de cáncer de mama, Pima y Boston). <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
</ol>

<h4>Libros de texto</h4>
<ul>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — Capítulo 4 para clasificadores bayesianos y k-NN, Capítulo 9 para SVM; el tratamiento más accesible y el complemento en R de este curso.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (MASS). 4th ed. Springer; 2002. — La referencia para las funciones de modelado de R que se usan aquí, incluida <code>class::knn</code>.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Capítulo 2.5 para la maldición de la dimensionalidad, 6.6.3 para Bayes ingenuo, 12 para SVM, 13 para k-NN. PDF gratuito.</li>
<li>Bishop CM. <em>Pattern Recognition and Machine Learning</em>. Springer; 2006. — 1.5 para teoría de la decisión, 2.5 para vecinos más cercanos, 7 para máquinas kernel dispersas, 8 para modelos gráficos. PDF gratuito.</li>
<li>Koller D, Friedman N. <em>Probabilistic Graphical Models: Principles and Techniques</em>. MIT Press; 2009. — La referencia exhaustiva sobre redes bayesianas, inferencia y aprendizaje.</li>
<li>Russell S, Norvig P. <em>Artificial Intelligence: A Modern Approach</em>. 4th ed. Pearson; 2021. — Capítulos 12–13 para probabilidad y redes bayesianas; Capítulo 19 para k-NN y SVM.</li>
</ul>

<h4>Recursos en línea</h4>
<ul>
<li>Las viñetas (vignettes) de los paquetes <code>e1071</code>, <code>class</code>, <code>mlbench</code> y <code>pROC</code> en CRAN: la documentación de cada función que se usa aquí.</li>
<li>LIBSVM y "A practical guide to support vector classification" — <a href="https://www.csie.ntu.edu.tw/~cjlin/libsvm/">csie.ntu.edu.tw/~cjlin/libsvm</a>; <code>e1071::svm</code> es su interfaz en R.</li>
<li>El paquete de R <code>bnlearn</code> y su repositorio de redes bayesianas, que incluye la red Asia completa con sus tablas publicadas — <a href="https://www.bnlearn.com/bnrepository/">bnlearn.com/bnrepository</a>.</li>
<li>UCI Machine Learning Repository — <a href="https://archive.ics.uci.edu">archive.ics.uci.edu</a>; fuente original de los datos de cáncer de mama.</li>
<li>Gigerenzer G. <em>Reckoning with Risk</em> (Penguin, 2002) y el Harding Center for Risk Literacy — <a href="https://www.hardingcenter.de">hardingcenter.de</a>; las frecuencias naturales en la comunicación médica.</li>
</ul>
